# 61 · Pathways that differ between human and mouse proximal tubule: choosing a continuous method, then comparing it with conventional analyses

**The question.** Which pathways differ between the human and the mouse proximal tubule (PT), and
does the difference depend on where along the tubule one looks?

**The data:**
- 12,272 PT cross-sections from 2 control mice and 2 cortex sections of 1 human donor, all male;
- 11,071 genes measured in both species;
- 1,513 Reactome, Hallmark and KEGG pathways.

**Two parts:**
- **Part 1 (section 2)** compares five continuous methods, which use the PT coordinate (SCF13). It
  chooses one of them by internal criteria applied in a fixed order.
- **Part 2 (section 3)** compares the chosen method with three conventional analyses: the whole PT,
  and S1/S2/S3 analysed two ways.

**The eight methods.** The numbers 1–7 are those used since notebook 61's seven-method version;
the per-segment method is 8.

| Group | # | Method | Code | What it compares |
|---|---|---|---|---|
| Continuous | 3 | T_spatial ranking, joint test | A6·JNT | the profile along the coordinate |
| Continuous | 7 | T_spatial ranking, preranked GSEA | A6·GSEA | the same ranking, GSEA's null |
| Continuous | 4 | ORA on split-plot significant genes | A6·ORAc | the profile, through a list of significant genes |
| Continuous | 6 | curve modules of the split-plot significant genes + ORA | B2·split | groups of genes with one difference shape |
| Continuous | 5 | curve modules of the top 10% T_spatial genes + ORA | B1·T10 | the same, from a ranking cutoff |
| Conventional reference | 1 | whole PT: pseudobulk DE + signed GSEA | A1·GSEA | the average over the whole PT |
| Conventional reference | 2 | S1/S2/S3 segment steps, joint test | A4·JNT | the profile over three zones; no coordinate |
| Conventional reference | 8 | S1/S2/S3 pseudobulk DE + ORA | A3·ORAc | the species difference within each zone |

The continuous methods form a 2 × 2 plus GSEA:

| | gene list from structure-level T_spatial ranks | gene list from the specimen-level split-plot test |
|---|---|---|
| **pathway-first** (test each pathway, then look at its genes) | 3 (and 7, same ranking) | 4 |
| **gene-first** (group the genes, then name the groups) | 5 (top 10%) | 6 |

**How every method is judged: the specimen relabeling.**
- **The species split** puts the two human sections against the two mice.
- **Each relabeling** pairs one human section with one mouse, which removes the species contrast
  and leaves only specimen differences.
- A method passes the **relabeling rule** when the relabelings call few pathways: mean relabeled
  calls ÷ species calls (NCDR) below 0.25, and each relabeling below 5% of tested pathways.

This notebook uses no external data. Notebook 63 holds the full grid of 45 methods.

**Assumptions.** SCF13 is treated as an exact coordinate. Specimens, not structures, are the
replicates. Results are descriptive. Every method is computed or loaded exactly as in notebook 63
and checked against its recorded values.

## 1 · Setup and shared helpers

In [ ]:
import argparse
import glob
import json
import os
import sys
import time
from pathlib import Path

NOTEBOOK_START = time.perf_counter()
start = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
project = next(p for p in (start, *start.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
nb12 = results_root / 'pt_pathway_remodeling_scfates'
nb31 = results_root / 'pt_pathway_method_selection'
nb37 = results_root / 'pt_pathway_final' / 'scfates'
nb45 = results_root / 'pt_revision_method' / 'tables'
nb45_cache = results_root / 'pt_revision_method' / 'stage_cache'
nb50 = results_root / 'pt_primary_pathways' / 'tables'
nb56 = results_root / 'pt_pathway_continuous' / '56_taxonomy'
nb57 = results_root / 'pt_pathway_continuous' / '57_beyond_steps' / 'scf13' / 'tables'
output = results_root / 'pt_pathway_methods_compared'
tables, figures = output / 'tables', output / 'figures'
for folder in (tables, figures):
    folder.mkdir(parents=True, exist_ok=True)
REQUIRED = [results_root / 'pt_reconstruction_v2' / 'primary_pt_coordinate.csv',
            nb31 / 'conventional_deseq2_all_partitions.csv', nb31 / 'conventional_signed_gsea_all_partitions.csv',
            nb12 / 'gene_statistics.csv', results_root / 'minimal_pt_scfates' / 'ortholog_map_used.csv',
            nb37 / 'tables' / 'matched_and_joint_tests_all_partitions.csv', nb37 / 'tables' / 'table_S1_strategy_specificity.csv',
            nb37 / 'tables' / 'table_S2_pathways.csv', nb37 / 'tables' / 'gene_level_statistics.csv',
            nb37 / 'tables' / 'segment_transitions_by_specimen.csv',
            nb45 / 'anchor_our_cohort.csv', nb50 / 'cohort_step_screen_per_pathway.csv', nb50 / 'primary_pathway_list.csv',
            nb56 / 'tables' / 'f_taxonomy.csv', nb56 / 'tables' / 'f_species_calls_by_approach.csv',
            nb57 / 'per_pathway_beyond_steps.csv',
            results_root / 'pt_revision_classes' / 'reviewed' / 'tables' / 'gene_classes_symmetric.csv']
for path in REQUIRED:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path} (run notebooks 12, 31, 37, 43, 45, 50, 56 and 57 first)')

NOTEBOOK_LOGIC_VERSION = '61.two_parts.1'      # this revision: two parts, eight methods, internal criteria only
CACHE_LOGIC_VERSION = '61.seven_methods.1'     # the cached stages are unchanged since this version; bump when one changes
ALPHA, NCDR_MAX, NULL_RATE = .05, .25, .05
LFC = 1.0                     # DE genes need |log2FC| >= 1 (notebook 12; method 8 and the worked-example rules)
K_MODULES, KM_STARTS, KM_SEED, TOP_SHARE = 8, 20, 56, .10   # notebook 56
K_RANGE, N_RESAMPLE = range(3, 9), 20                       # module-count selection (notebook 56)
CALIBRATION_MAX = 1.5 * ALPHA  # criterion 2: share of relabeled p <= 0.05 at most 0.075 (set at this revision)
EXAMPLE_BINS = 12             # specimen x coordinate-bin means in the worked examples (notebook 41)
N_JOBS = int(os.environ.get('PSEUDOSPACE_N_JOBS', '32'))  # parallel workers; never part of a cache key

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from joblib import Parallel, delayed, parallel_config
from scipy import sparse
from scipy.stats import hypergeom, norm
from statsmodels.stats.multitest import multipletests
from threadpoolctl import threadpool_limits

MM = 1 / 25.4
HUMAN, MOUSE, NULL, LIGHT, INK, MUTED, ACCENT = '#D55E00', '#0072B2', '#9A9A9A', '#D9D9D9', '#222222', '#666666', '#009E73'
SPECIMENS = {'mouse': ['Ctrl1A2', 'Ctrl1A4'], 'human': ['HUK1_COR1', 'HUK1_MED1']}
SPECIMEN_MARKER = {'Ctrl1A2': 'o', 'Ctrl1A4': 's', 'HUK1_COR1': 'o', 'HUK1_MED1': 's'}
# One colour per method in every figure (a validated categorical palette); method 5 fails the relabeling rule and is drawn lighter.
METHOD_COLOR = {'A6·JNT': '#2a78d6', 'A6·GSEA': '#1baf7a', 'A6·ORAc': '#eb6834', 'B2·split': '#eda100', 'B1·T10': '#9C93D3',
                'A1·GSEA': '#008300', 'A4·JNT': '#e87ba4', 'A3·ORAc': '#4a3aa7'}
plt.rcParams.update({
    'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 6.5, 'axes.titlesize': 6.5,
    'axes.titleweight': 'bold', 'axes.labelsize': 6.5, 'xtick.labelsize': 6, 'ytick.labelsize': 6,
    'legend.fontsize': 6, 'legend.frameon': False, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': .6, 'xtick.major.width': .6, 'ytick.major.width': .6, 'xtick.major.size': 2.5,
    'ytick.major.size': 2.5, 'lines.linewidth': 1.0, 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})
pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_colwidth', 70, 'display.max_rows', 80)
print('Results folder:', output, '· parallel workers:', N_JOBS)

### 1.1 · Structures, genes, pathways, partitions and shared helpers

The construction is notebook 37's on SCF13:
- the structures inside every specimen's 1st–99th percentile range of the coordinate;
- the genes measured in both species and detected in ≥ 2% of structures;
- the pathways with 10–300 tested genes.

In [ ]:
from pseudospace.coordinate_inputs import pt_inputs, read_coordinate
from pseudospace.pathway_calibration import balanced_partitions
from pseudospace.pathway_decoys import joint_matched_test
from pseudospace.stage_cache import cached_frame, cached_payload, digest
from pseudospace.stats_gam import gam_internal_knots, make_gam_design

inputs = pt_inputs(results_root, data_root, read_coordinate(results_root / 'pt_reconstruction_v2' / 'primary_pt_coordinate.csv'))
position, human, specimen, segment = (inputs[k] for k in ('position', 'human', 'specimen', 'segment'))
genes, gene_sets, strata = inputs['genes'], inputs['gene_sets'], inputs['strata']
pathways = list(gene_sets)
partitions = balanced_partitions(specimen, np.where(human, 'human', 'mouse'))
SWAPS = [p for p in partitions if p != 'species']
PARTS = ['species', *SWAPS]
ALL_SPECIMENS = SPECIMENS['mouse'] + SPECIMENS['human']
assert len(position) == 12272 and len(genes) == 11071 and len(pathways) == 1513, 'notebook 37 universe changed'
table37 = pd.read_csv(nb37 / 'tables' / 'table_S2_pathways.csv').set_index('pathway_id')
NAMES = table37.name.reindex(pathways)
VIF = table37.vif.reindex(pathways)
spec37 = pd.read_csv(nb37 / 'tables' / 'table_S1_strategy_specificity.csv').set_index('strategy')
gene37 = pd.read_csv(nb37 / 'tables' / 'gene_level_statistics.csv', index_col=0).T_spatial.reindex(genes)
cache = output / 'stage_cache'
code = digest([CACHE_LOGIC_VERSION])
input_key = {'Y': digest(inputs['Y']), 'position': position, 'specimen': specimen, 'genes': genes}
grid = inputs['grid']
lo, hi = grid[0], grid[-1]
knots = gam_internal_knots(position, basis_df=6)
print('Partitions:', partitions)

CALLS, EVIDENCE, METHODS = {}, {}, []
SPECIES_Q, RELABEL_P = {}, {}   # per-pathway BH q (species split) and p (each relabeling), per method
RETURNS = ['yes/no', 'direction', 'location', 'shape', 'genes']
LOADED = 'loaded'


def register(key, number, label, calls, *, evidence=None, returns=('yes/no',), runtime=None, note=''):
    """Record one method: its calls under the species split and both relabelings, and the specificity rule."""
    CALLS[key] = {p: set(calls[p]) for p in PARTS}
    if evidence is not None:
        EVIDENCE[key] = pd.Series(evidence, dtype=float).reindex(pathways)
    relabels = [len(CALLS[key][s]) for s in SWAPS]
    n_species = len(CALLS[key]['species'])
    ncdr = np.mean(relabels) / n_species if n_species else 0.
    passes = bool(n_species and ncdr < NCDR_MAX and max(relabels) < NULL_RATE * len(pathways))
    METHODS.append({'key': key, '#': number, 'method': label, 'species calls': n_species,
                    'relabeled calls': ', '.join(map(str, relabels)), 'NCDR': ncdr, 'specific': passes,
                    'runtime (s)': runtime, **{r: r in returns for r in RETURNS}, 'note': note})
    print(f"{number} · {key:<9} {n_species:>4} species calls; relabeled {relabels}; {'passes' if passes else 'fails'} the relabeling rule")


def timed(stage, compute, kind='frame', **cache_kwargs):
    """Cache a single (serial) computation together with its uncached wall time; returns (result, seconds)."""
    def wrapped():
        t0 = time.perf_counter()
        out = compute()
        seconds = time.perf_counter() - t0
        if kind == 'frame':
            return out.assign(__seconds=seconds)
        out = dict(out)
        out['__seconds'] = np.array(seconds)
        return out
    if kind == 'frame':
        result = cached_frame(stage, wrapped, root=cache, code=code, **cache_kwargs)
        return result.drop(columns='__seconds'), float(result['__seconds'].iloc[0])
    result = cached_payload(stage, wrapped, root=cache, code=code, **cache_kwargs)
    return {k: v for k, v in result.items() if k != '__seconds'}, float(result['__seconds'])


def _run_task(function, args):
    t0 = time.perf_counter()
    with threadpool_limits(1):
        out = function(*args)
    return out, time.perf_counter() - t0


def fan_out(function, task_args, n_jobs=None):
    """``function(*args)`` for every args tuple, in task order, on ``n_jobs`` loky workers (one BLAS thread each).

    Random tasks carry their own seeds, derived from fixed task ids, so results do not depend on ``n_jobs``.
    """
    jobs = N_JOBS if n_jobs is None else n_jobs
    if jobs == 1 or len(task_args) == 1:
        pairs = [_run_task(function, a) for a in task_args]
    else:
        with parallel_config(backend='loky', inner_max_num_threads=1):
            pairs = Parallel(n_jobs=min(jobs, len(task_args)))(delayed(_run_task)(function, a) for a in task_args)
    return [p[0] for p in pairs], [p[1] for p in pairs]


def cached_tasks(stage, function, tasks, **cache_kwargs):
    """Fan out ``tasks`` (task id -> args) and cache all results as one stage-cache entry (key never includes n_jobs).

    Returns {task id: (result dict, seconds)}; seconds are each task's own compute time.
    """
    ids = list(tasks)

    def compute():
        results, seconds = fan_out(function, [tasks[i] for i in ids])
        payload = {}
        for i, result, sec in zip(ids, results, seconds):
            payload.update({f'{i}|{k}': np.asarray(v) for k, v in result.items()})
            payload[f'{i}|__seconds'] = np.array(sec)
        return payload
    stored = cached_payload(stage, compute, root=cache, code=code, **cache_kwargs)
    out = {i: {} for i in ids}
    for key, value in stored.items():
        i, k = key.rsplit('|', 1)
        out[i][k] = value
    return {i: (out[i], float(out[i].pop('__seconds'))) for i in ids}


def slug(p):
    return p.replace(':', '_').replace('+', '_')


def membership(sets, universe):
    index = pd.Index(universe)
    rows, cols = [], []
    for i, members in enumerate(sets.values()):
        idx = index.get_indexer(members)
        idx = idx[idx >= 0]
        rows.extend([i] * len(idx)), cols.extend(idx)
    return sparse.csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(sets), len(index)))


member_matrix = membership(gene_sets, genes)


def ora_pvalues(selected, matrix):
    """One-sided hypergeometric over-representation p for a boolean gene selection."""
    selected = np.asarray(selected, bool)
    if not selected.any():
        return np.ones(matrix.shape[0])
    k = np.asarray(matrix @ selected.astype(float)).ravel()
    return hypergeom.sf(k - 1, len(selected), np.asarray(matrix.sum(axis=1)).ravel(), selected.sum())


def bh(p):
    return multipletests(np.nan_to_num(np.asarray(p, float), nan=1.), method='fdr_bh')[1]


def newest(folder, stage, kind='npz'):
    files = glob.glob(str(folder / f'{stage}__*.{kind}'))
    if not files:
        raise FileNotFoundError(f'{stage} not found in {folder}')
    files.sort(key=lambda p: json.loads(Path(p + '.key.json').read_text())['written_utc'])
    if kind == 'csv':
        return pd.read_csv(files[-1])
    with np.load(files[-1]) as stored:
        return {k: stored[k] for k in stored.files}


def check_counts(key, strategy):
    row = spec37.loc[strategy]
    assert [len(CALLS[key][p]) for p in PARTS] == [row.species_calls, row.relabel_calls_Ctrl1A2, row.relabel_calls_Ctrl1A4], \
        (key, [len(CALLS[key][p]) for p in PARTS])

## 2 · Part 1: the continuous methods

Five methods use the PT coordinate. They are described one by one (2.1–2.4), compared side by
side (2.5), and one is chosen by fixed internal criteria (2.6).

### 2.1 · Method 3: continuous, pathway-first, rank-based: T_spatial with the joint test (A6·JNT)

**What it does.**
- For each gene, a 6-df spline along the coordinate is fitted in each species, and **T_spatial**
  is the partial F of the species × position terms beyond a species offset. Each species gets half
  the weight and each specimen an equal share.
- All genes are ranked by T_spatial, and the pathway is tested with the same joint test as method 2.

**One call means:** the shape of the pathway genes' expression along the PT depends on species more
than for matched genes.

**It also returns:**
- the pathway's difference curve Δ(s), human minus mouse, along the coordinate;
- where Δ(s) peaks, and the direction there;
- the genes that carry the signal.

**What it cannot tell you:**
- an average offset, which is removed on purpose;
- whether a single gene is significant. T_spatial ranks genes; its degrees of freedom count
  structures, so it is not a calibrated gene test.

The results are notebook 37's, loaded and checked; the joint z is recomputed.

In [ ]:
tests37 = pd.read_csv(nb37 / 'tables' / 'matched_and_joint_tests_all_partitions.csv')
tests37 = tests37[tests37.partition.isin(PARTS)]
recomputed = joint_matched_test(tests37.drop(columns=['z_matched', 'vif', 'z_joint', 'p_joint', 'q_joint']), VIF)
np.testing.assert_allclose(recomputed.z_joint, tests37.z_joint, rtol=1e-9)   # Guard: joint z = matched z ÷ √VIF
spatial37 = tests37[tests37.statistic.eq('T_spatial')]
register('A6·JNT', 3, 'Continuous, pathway-first, rank-based: T_spatial, joint test',
         {p: spatial37.pathway_id[spatial37.partition.eq(p) & spatial37.effect.gt(0) & spatial37.q_joint.le(ALPHA)] for p in PARTS},
         evidence=spatial37[spatial37.partition.eq('species')].set_index('pathway_id').z_joint,
         returns=('yes/no', 'direction', 'location', 'shape', 'genes'), runtime=LOADED)
check_counts('A6·JNT', '8 · Gene GAM × smooth position (T_spatial) (joint test)')
by_part = {p: spatial37[spatial37.partition.eq(p)].set_index('pathway_id').reindex(pathways) for p in PARTS}
SPECIES_Q['A6·JNT'] = by_part['species'].q_joint.to_numpy(float)
RELABEL_P['A6·JNT'] = {s: by_part[s].p_joint.to_numpy(float) for s in SWAPS}
curves = {p: newest(nb56 / 'stage_cache', 'curves_' + slug(p)) for p in PARTS}   # Δ(s) per gene (notebook 56)
np.testing.assert_allclose(curves['species']['T_spatial'], gene37, rtol=1e-6)    # Guard: notebook 37's T_spatial

### 2.2 · Method 7: continuous, pathway-first, rank-based with GSEA: preranked GSEA on T_spatial (A6·GSEA)

**What it does.** It uses the same T_spatial ranking as method 3, but tests each pathway with
preranked GSEA (weight 0, multilevel), the most common set test.

**One call means:** the pathway's genes gather near the top of the T_spatial ranking more than
random gene sets of the same size do.

**Why it is less selective than the joint test.** GSEA calls about 20% of pathways where the joint
test calls 4%, and section 2.7 shows that its p-values are too small under relabeling. The joint
test of method 3 adds two corrections that GSEA lacks:
- **Matched background.** GSEA compares a pathway with random gene sets drawn from all genes. The
  joint test draws them from genes matched on expression, detection and coverage. Highly expressed,
  well-detected genes give the spline model more power, so they rank higher on T_spatial, and
  pathways made of such genes look enriched to GSEA.
- **Gene correlation.** GSEA treats a pathway's genes as independent. Co-regulated genes move
  together, so their ranks carry less independent evidence than their number suggests. The joint
  test divides the z-score by √VIF, the variance inflation estimated from the within-specimen
  correlation of the pathway's genes.

**What it cannot tell you:** the same as method 3, and its calls cannot be ranked by an effect that
accounts for these two biases.

The calls are notebook 31's, loaded and checked against notebook 37's counts. The per-pathway NES and p
are notebook 37's saved GSEA runs; they reproduce the calls exactly.

In [ ]:
hits31 = pd.read_csv(nb31 / 'strategy_hits_all_partitions.csv')
gsea_spatial = hits31[hits31.strategy.eq('8c · T_spatial, preranked GSEA')]
spatial_gsea37 = newest(nb37 / 'stage_cache', 'spatial_gsea_species', kind='csv').set_index('pathway_id')
register('A6·GSEA', 7, 'Continuous, pathway-first, rank-based: T_spatial, preranked GSEA',
         {p: gsea_spatial.pathway_id[gsea_spatial.partition.eq(p) & gsea_spatial.hit] for p in PARTS},
         evidence=-np.log10(spatial_gsea37.p.clip(lower=1e-300)), returns=('yes/no', 'direction', 'location', 'shape', 'genes'),
         runtime=LOADED)
check_counts('A6·GSEA', '8c · T_spatial, preranked GSEA')
for p in PARTS:   # per-pathway values; enrichment at the top of the ranking only (NES > 0), as the calls
    g = newest(nb37 / 'stage_cache', f'spatial_gsea_{slug(p)}', kind='csv').set_index('pathway_id').reindex(pathways)
    q = bh(g.p)
    assert set(np.array(pathways)[(g.NES > 0).to_numpy() & (q <= ALPHA)]) == CALLS['A6·GSEA'][p], p   # Guard: same calls
    if p == 'species':
        SPECIES_Q['A6·GSEA'] = np.where(g.NES > 0, q, 1.)
    else:   # one-sided (upper-tail) p from GSEA's nominal p, which is computed within the sign of the enrichment score
        RELABEL_P.setdefault('A6·GSEA', {})[p] = np.where(g.NES > 0, g.p / 2, 1 - g.p / 2)

### 2.3 · Method 4: continuous, pathway-first, threshold-based: ORA on split-plot significant genes (A6·ORAc)

**What it does.**
1. **Each gene is tested at specimen level.** This is the split plot of notebook 37:
   - expression is averaged in specimen × 8-bin cells along the coordinate;
   - the species × position interaction is judged against specimen × position variation, with a
     moderated F.
2. Genes with BH q ≤ 0.05 form the list. With the species split this is 4,588 genes; with either
   relabeling it is none.
3. ORA (a hypergeometric test) asks whether a pathway holds more listed genes than chance.

**One call means:** the pathway is over-represented among genes whose positional profile differs
between species, judged against the specimens.

**What it cannot tell you:**
- how strongly each gene differs. The list is yes or no, and the 4,588 genes weigh equally;
- anything about pathways whose genes change only modestly.

Its gene list is the same one that method 6 clusters.

In [ ]:
split37 = {p: newest(nb37 / 'stage_cache', f'split_plot_{slug(p)}') for p in PARTS}
SPLIT_GENES = {p: bh(split37[p]['p']) <= ALPHA for p in PARTS}
print('Split-plot significant genes (BH ≤ 0.05):', {p: int(v.sum()) for p, v in SPLIT_GENES.items()})
t0 = time.perf_counter()
ora_split = {p: ora_pvalues(SPLIT_GENES[p], member_matrix) for p in PARTS}
register('A6·ORAc', 4, 'Continuous, pathway-first, threshold-based: split-plot significant genes + ORA',
         {p: np.array(pathways)[bh(v) <= ALPHA] for p, v in ora_split.items()},
         evidence=-np.log10(pd.Series(ora_split['species'], index=pathways)),
         returns=('yes/no', 'direction', 'location', 'shape', 'genes'), runtime=time.perf_counter() - t0)
SPECIES_Q['A6·ORAc'] = bh(ora_split['species'])
RELABEL_P['A6·ORAc'] = {s: ora_split[s] for s in SWAPS}

### 2.4 · Methods 6 and 5: continuous, gene-first: curve modules + ORA

**What they do.**
1. Take a list of genes and each gene's difference curve Δ(s), human minus mouse along the
   coordinate, centred and scaled.
2. Group the curves by shape with k-means.
3. Ask with ORA which pathways are over-represented in each group (module). A pathway is called when
   it is enriched in any module.

The two methods differ only in the gene list:
- **Method 5, naive (B1·T10):** the top 10% of genes by T_spatial, with k = 8 (notebook 56's
  stability choice).
- **Method 6, specimen-aware (B2·split):** the split-plot significant genes of method 4. k is chosen
  among 3–8 by resampling stability on the species run.

The relabelings recluster their own lists. A relabeling with fewer than 10·k significant genes has
no modules and no calls.

**One call means:** the pathway's genes are over-represented in a group of genes that share one
shape of species difference.

**What they cannot tell you:**
- whether the pathway as a whole differs. A pathway whose genes change in two different ways is
  split across modules;
- anything beyond the list. ORA here is not matched on expression, detection or coverage.

**What they can show:** a group of genes with one shape of species difference, such as method 6's
module of genes whose difference falls toward S3 (section 5).

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score


def standardize(d):
    return (d - d.mean(axis=1, keepdims=True)) / np.maximum(d.std(axis=1, keepdims=True), 1e-12)


def kmeans_task(data, k, seed, pick):
    """One k-means fit (20 starts) on all rows (``pick`` empty) or on the picked rows."""
    rows = data if not len(pick) else data[pick]
    return {'labels': KMeans(n_clusters=k, n_init=KM_STARTS, random_state=seed).fit(rows).labels_}


def choose_k(stage, data):
    """Notebook 56's rule: mean ARI of 20 resamples of 80% per k; the most stable k, ties to the smaller."""
    rng = np.random.default_rng(KM_SEED)
    tasks = {}
    for k in K_RANGE:
        tasks[f'{k}|full'] = (data, k, KM_SEED, np.array([], int))
        for r in range(N_RESAMPLE):
            tasks[f'{k}|{r}'] = (data, k, KM_SEED + r + 1, np.sort(rng.choice(len(data), int(.8 * len(data)), replace=False)))
    runs = cached_tasks(stage, kmeans_task, tasks, params={'k': list(K_RANGE), 'resamples': N_RESAMPLE, 'seed': KM_SEED,
                                                           'starts': KM_STARTS}, inputs={'data': digest(data)})
    stability = {k: float(np.mean([adjusted_rand_score(runs[f'{k}|full'][0]['labels'][tasks[f'{k}|{r}'][3]],
                                                       runs[f'{k}|{r}'][0]['labels']) for r in range(N_RESAMPLE)]))
                 for k in K_RANGE}
    return max(stability, key=lambda k: (round(stability[k], 6), -k)), stability, sum(sec for _, sec in runs.values())


MODULES, BEST_P = {}, {}


def curve_modules(key, number, label, data_of, k_fixed=None, min_per_module=0):
    """Modules of each partition's selected genes, annotated by ORA; calls = pathways enriched in any module."""
    if k_fixed is None:
        k, stability, sec_k = choose_k(f'kselect_{key[:2]}', data_of('species')[1])
        print(f'{key}: mean resample ARI by k', {kk: round(v, 3) for kk, v in stability.items()}, '→ k =', k)
    else:
        k, sec_k = k_fixed, 0.
    prepared = {p: data_of(p) for p in PARTS}
    fit = {p: v for p, v in prepared.items() if len(v[0]) >= max(min_per_module * k, k + 1)}
    runs = cached_tasks(f'modules_{key[:2]}', kmeans_task, {p: (fit[p][1], k, KM_SEED, np.array([], int)) for p in fit},
                        params={'k': k, 'seed': KM_SEED, 'starts': KM_STARTS}, inputs={p: digest(fit[p][1]) for p in fit})
    calls, evidence = {}, None
    t0 = time.perf_counter()
    for p in PARTS:
        if p not in fit:
            calls[p] = []
            BEST_P[(key, p)] = np.ones(len(pathways))   # no modules, so no test
            print(f'{key} {p}: {len(prepared[p][0])} genes selected, fewer than {min_per_module} × k, so no modules')
            continue
        rows, labels = fit[p][0], runs[p][0]['labels']
        pv = np.column_stack([ora_pvalues(np.isin(np.arange(len(genes)), rows[labels == m]), member_matrix) for m in range(k)])
        q = bh(pv.ravel()).reshape(pv.shape)
        calls[p] = np.array(pathways)[(q <= ALPHA).any(axis=1)]
        BEST_P[(key, p)] = np.minimum(1, k * pv.min(axis=1))   # best module, Bonferroni over the k modules
        if p == 'species':
            SPECIES_Q[key] = q.min(axis=1)                     # best module's q (one BH over module x pathway)
        if p == 'species':
            evidence = pd.Series(-np.log10(pv.min(axis=1).clip(1e-300)), index=pathways)
            MODULES[key] = {'genes': [list(np.array(genes)[rows[labels == m]]) for m in range(k)], 'q': q,
                            'centroids': np.array([fit[p][1][labels == m].mean(axis=0) for m in range(k)])}
    seconds = sec_k + sum(sec for _, sec in runs.values()) + time.perf_counter() - t0
    register(key, number, label, calls, evidence=evidence, returns=('yes/no', 'shape', 'genes'), runtime=seconds, note=f'k = {k}')


def top_rows(values, share=TOP_SHARE):
    return np.sort(np.argsort(-values)[:int(round(share * len(values)))])


k_b1, _, _ = choose_k('kselect_B1', standardize(curves['species']['delta'][top_rows(curves['species']['T_spatial'])]))
assert k_b1 == K_MODULES, 'notebook 56 chose k = 8'
curve_modules('B1·T10', 5, 'Continuous, gene-first, naive: curve modules of the top 10% T_spatial genes + ORA',
              lambda p: (top_rows(curves[p]['T_spatial']), standardize(curves[p]['delta'][top_rows(curves[p]['T_spatial'])])),
              k_fixed=K_MODULES)
row56 = pd.read_csv(nb56 / 'tables' / 'f_taxonomy.csv').set_index('approach').loc[f'Curve modules (k = {K_MODULES}) + ORA']
assert [len(CALLS['B1·T10'][p]) for p in PARTS] == [int(row56['species calls'])] + [int(x) for x in row56['relabeled calls'].split(', ')]   # Guard: notebook 56
curve_modules('B2·split', 6, 'Continuous, gene-first, specimen-aware: curve modules of the split-plot significant genes + ORA',
              lambda p: (np.flatnonzero(SPLIT_GENES[p]), standardize(curves[p]['delta'][SPLIT_GENES[p]])), min_per_module=10)
for key in ('B1·T10', 'B2·split'):
    RELABEL_P[key] = {s: BEST_P[(key, s)] for s in SWAPS}

Parallel execution check: k-means fits give identical outputs with one worker and with `N_JOBS`
workers.

In [ ]:
check_args = [(standardize(curves['species']['delta'][top_rows(curves['species']['T_spatial'])]), 5, KM_SEED + r,
               np.sort(np.random.default_rng(r).choice(1107, 885, replace=False))) for r in range(3)]
serial, _ = fan_out(kmeans_task, check_args, n_jobs=1)
parallel, _ = fan_out(kmeans_task, check_args, n_jobs=N_JOBS)
assert all(np.array_equal(a['labels'], b['labels']) for a, b in zip(serial, parallel))
print(f'Parallel check passed: n_jobs = 1 and n_jobs = {N_JOBS} give identical k-means labels')

### 2.5 · The five continuous methods side by side

**Lower panels.** Each dot is a pathway: −log10 BH q of the column method (x) against the row
method (y).
- Dashed lines mark q = 0.05.
- Black: called by both methods. Method colour: called by that method only. Grey: neither.
- Grey bands hold the pathways with q ≥ 0.99, jittered, so none is dropped.
- GSEA counts enrichment at the top of the ranking only (NES > 0), as its calls do. Its axis is
  capped at 10, and pathways beyond the cap are drawn as triangles on the edge.

**Diagonal.** Each method's calls under the species split and under the two relabelings.

**Upper panels.** The overlap of the two methods' calls, shaded by the Jaccard index.
- The bar reads row only | both | column only.
- Spearman of −log10 q over all 1,513 pathways is in small print. The many ties at q = 1 lower it
  for the ORA-based methods (4, 6, 5).

In [ ]:
from matplotlib.colors import to_rgb


def lighter(color, amount=.5):
    r, g, b = to_rgb(color)
    return (r + (1 - r) * amount, g + (1 - g) * amount, b + (1 - b) * amount)


CONTINUOUS = ['A6·JNT', 'A6·GSEA', 'A6·ORAc', 'B2·split', 'B1·T10']   # rank-based first, then gene-list-based
METHOD_ROW = {m['key']: m for m in METHODS}
SHORT_NAME = {'A6·JNT': 'Joint test on T_spatial', 'A6·GSEA': 'GSEA on T_spatial', 'A6·ORAc': 'ORA on split-plot genes',
              'B2·split': 'Modules of split-plot genes', 'B1·T10': 'Modules of top-10% genes', 'A1·GSEA': 'Whole PT, GSEA',
              'A4·JNT': 'S1/S2/S3 steps', 'A3·ORAc': 'S1/S2/S3 DE + ORA'}
NUMBER = {m['key']: m['#'] for m in METHODS}
GSEA_CAP, BAND = 10., (-.3, -.06)
pair_rng = np.random.default_rng(11)


def neglog_q(key, cap=None):
    """−log10 q with q >= 0.99 jittered into the band below zero, and values beyond ``cap`` set to the cap."""
    v = -np.log10(np.asarray(SPECIES_Q[key], float))
    out = v.copy()
    pile = v < -np.log10(.99)
    out[pile] = pair_rng.uniform(*BAND, pile.sum())
    over = np.zeros(len(v), bool) if cap is None else v > cap
    out[over] = cap if cap is not None else out[over]
    return out, over


VALUES = {k: neglog_q(k, GSEA_CAP if k == 'A6·GSEA' else None) for k in CONTINUOUS}
LIMITS = {k: (BAND[0] - .08, (GSEA_CAP if k == 'A6·GSEA' else max(VALUES[k][0].max(), -np.log10(ALPHA))) * 1.06) for k in CONTINUOUS}
IS_CALLED = {k: np.isin(pathways, list(CALLS[k]['species'])) for k in METHOD_ROW}
for k in CONTINUOUS:   # Guard: the per-pathway q reproduces each method's calls
    assert set(np.array(pathways)[np.asarray(SPECIES_Q[k]) <= ALPHA]) == CALLS[k]['species'], k
rho = pd.DataFrame({k: -np.log10(SPECIES_Q[k]) for k in CONTINUOUS}).corr(method='spearman')
THR = -np.log10(ALPHA)
fig = plt.figure(figsize=(180 * MM, 180 * MM))
gs = fig.add_gridspec(5, 5, hspace=.14, wspace=.14, left=.08, right=.99, top=.99, bottom=.03)
pair_rows = []
for i, ky in enumerate(CONTINUOUS):
    for j, kx in enumerate(CONTINUOUS):
        ax = fig.add_subplot(gs[i, j])
        if i == j:
            row = METHOD_ROW[ky]
            ax.set_facecolor(lighter(METHOD_COLOR[ky], .82))
            ax.text(.5, .66, str(NUMBER[ky]), ha='center', va='center', fontsize=15, fontweight='bold', color=METHOD_COLOR[ky],
                    transform=ax.transAxes)
            ax.text(.5, .4, SHORT_NAME[ky], ha='center', va='center', fontsize=6, transform=ax.transAxes)
            ax.text(.5, .22, f"{row['species calls']} calls; relabeled {row['relabeled calls']}", ha='center', va='center',
                    fontsize=5.6, transform=ax.transAxes)
            if not row['specific']:
                ax.text(.5, .07, 'fails relabeling', ha='center', va='center', fontsize=5.6, color='#C0392B', fontweight='bold',
                        transform=ax.transAxes)
            ax.set_xticks([]), ax.set_yticks([])
            for side in ax.spines.values():
                side.set_visible(False)
        elif i < j:
            a, b = IS_CALLED[ky], IS_CALLED[kx]
            both, row_only, col_only = int((a & b).sum()), int((a & ~b).sum()), int((~a & b).sum())
            jac = both / max(both + row_only + col_only, 1)
            pair_rows.append({'row': f'{NUMBER[ky]} {ky}', 'column': f'{NUMBER[kx]} {kx}', 'both': both, 'row only': row_only,
                              'column only': col_only, 'Jaccard': jac, 'Spearman': rho.loc[ky, kx]})
            ax.set_facecolor(plt.get_cmap('Greys')(.06 + .55 * jac))
            text = 'white' if jac > .55 else INK
            ax.text(.5, .78, f'Jaccard {jac:.2f}', ha='center', va='center', fontsize=7.5, fontweight='bold', color=text,
                    transform=ax.transAxes)
            left, total = .08, max(both + row_only + col_only, 1)
            for width, color in ((row_only, METHOD_COLOR[ky]), (both, INK), (col_only, METHOD_COLOR[kx])):
                ax.add_patch(plt.Rectangle((left, .42), .84 * width / total, .16, transform=ax.transAxes, color=color, lw=0))
                left += .84 * width / total
            ax.text(.08, .3, f'{NUMBER[ky]} only {row_only}', ha='left', va='center', fontsize=5.6, color=text, transform=ax.transAxes)
            ax.text(.92, .3, f'{NUMBER[kx]} only {col_only}', ha='right', va='center', fontsize=5.6, color=text, transform=ax.transAxes)
            ax.text(.5, .19, f'both {both}', ha='center', va='center', fontsize=5.6, color=text, transform=ax.transAxes)
            ax.text(.5, .06, f'Spearman {rho.loc[ky, kx]:.2f}', ha='center', va='center', fontsize=4.8,
                    color='white' if jac > .55 else MUTED, transform=ax.transAxes)
            ax.set_xticks([]), ax.set_yticks([])
            for side in ax.spines.values():
                side.set_visible(False)
        else:
            (x, xo), (y, yo) = VALUES[kx], VALUES[ky]
            a, b = IS_CALLED[kx], IS_CALLED[ky]
            colors = np.array([INK if p and q else METHOD_COLOR[kx] if p else METHOD_COLOR[ky] if q else LIGHT
                               for p, q in zip(a, b)], dtype=object)
            order = np.argsort((a | b).astype(int), kind='stable')
            capped = xo | yo
            sel = order[~capped[order]]
            ax.scatter(x[sel], y[sel], s=4, c=list(colors[sel]), linewidths=0, zorder=2)
            for flag, marker in ((xo & ~yo, '>'), (yo & ~xo, '^'), (xo & yo, 'D')):
                if flag.any():
                    ax.scatter(x[flag], y[flag], s=9, c=list(colors[flag]), marker=marker, linewidths=0, zorder=3)
            ax.axvline(THR, color=MUTED, lw=.5, ls='--', zorder=1)
            ax.axhline(THR, color=MUTED, lw=.5, ls='--', zorder=1)
            ax.axvspan(BAND[0] - .03, BAND[1] + .03, color='#EEEEEE', lw=0, zorder=0)
            ax.axhspan(BAND[0] - .03, BAND[1] + .03, color='#EEEEEE', lw=0, zorder=0)
            ax.set_xlim(*LIMITS[kx])
            ax.set_ylim(*LIMITS[ky])
            if j:
                ax.set_yticklabels([])
            if i != len(CONTINUOUS) - 1:
                ax.set_xticklabels([])
        if j == 0 and i > 0:
            ax.set_ylabel(f'{NUMBER[ky]}  −log10 q', color=METHOD_COLOR[ky])
        if i == len(CONTINUOUS) - 1 and j < len(CONTINUOUS) - 1:
            ax.set_xlabel(f'{NUMBER[kx]}  −log10 q', color=METHOD_COLOR[kx])
for ext in ('pdf', 'png'):
    fig.savefig(figures / f'fig1_continuous_pairwise.{ext}', bbox_inches='tight')
plt.show()
PAIRS = pd.DataFrame(pair_rows)
PAIRS.to_csv(tables / 'continuous_pairwise_overlap.csv', index=False)
display(PAIRS.round(2))

### 2.6 · Choosing one continuous method

Four criteria are applied in order. Each removes methods before the next is looked at.

1. **The relabeling rule.** NCDR < 0.25, and each relabeling calls < 5% of the tested pathways.
2. **A calibrated set test.** Pooled over the two relabelings (2 × 1,513 p-values), the share of
   p ≤ 0.05 is at most 1.5 × the nominal 5%, that is 0.075.
   - Under a relabeling there is no species difference, so a calibrated test gives about 5% of
     p ≤ 0.05.
   - **This threshold was set by us at this revision, after the relabeling results were known.**
3. **The most species calls.**
4. **What a call returns** (direction, location, shape, genes), and whether a gene cutoff or a
   number of modules k must be chosen.

The p-values used for criterion 2 are those behind each method's calls:
- method 3: the joint test's one-sided p;
- method 7: GSEA's nominal p, made one-sided for enrichment at the top (p/2 when NES > 0, else 1 − p/2);
- methods 4 and 6: no gene passes the split-plot test under either relabeling. Method 4's ORA has
  no listed genes and method 6 has no modules, so every p is 1;
- method 5: the best module's ORA p, multiplied by its k = 8 modules (Bonferroni).

**a.** The calibration plot: observed against expected −log10 p for the pooled relabelings. A line
above the diagonal has more small p-values than chance.

**b.** The decision table.

In [ ]:
n_pool = 2 * len(pathways)
expected = -np.log10((np.arange(1, n_pool + 1) - .5) / n_pool)
CALIBRATION = {k: float(np.mean(np.concatenate([RELABEL_P[k][s] for s in SWAPS]) <= ALPHA)) for k in CONTINUOUS}
NEEDS = {'A6·JNT': 'no', 'A6·GSEA': 'no', 'A6·ORAc': 'gene cutoff', 'B2·split': 'gene cutoff + k', 'B1·T10': 'ranking cutoff + k'}
remaining, decision = list(CONTINUOUS), {}
for k in CONTINUOUS:                                    # criterion 1
    if not METHOD_ROW[k]['specific']:
        decision[k] = f"out at 1: relabelings call {METHOD_ROW[k]['relabeled calls']}"
        remaining.remove(k)
for k in list(remaining):                               # criterion 2
    if CALIBRATION[k] > CALIBRATION_MAX:
        decision[k] = f'out at 2: {CALIBRATION[k]:.3f} of relabeled p ≤ 0.05'
        remaining.remove(k)
CHOSEN = max(remaining, key=lambda k: METHOD_ROW[k]['species calls'])   # criterion 3
for k in remaining:
    decision[k] = ('chosen: most calls among those left' if k == CHOSEN else
                   f"out at 3: {METHOD_ROW[k]['species calls']} calls against {METHOD_ROW[CHOSEN]['species calls']}")
DECISION = pd.DataFrame({
    '#': [NUMBER[k] for k in CONTINUOUS], 'method': [SHORT_NAME[k] for k in CONTINUOUS],
    '1 relabeled calls (NCDR)': [f"{'pass' if METHOD_ROW[k]['specific'] else 'fail'}: {METHOD_ROW[k]['relabeled calls']} "
                                 f"({METHOD_ROW[k]['NCDR']:.2f})" for k in CONTINUOUS],
    '2 share of relabeled p ≤ 0.05': [f"{'pass' if CALIBRATION[k] <= CALIBRATION_MAX else 'fail'}: {CALIBRATION[k]:.3f}"
                                     for k in CONTINUOUS],
    '3 species calls': [METHOD_ROW[k]['species calls'] for k in CONTINUOUS],
    **{r: ['✓' if METHOD_ROW[k][r] else '' for k in CONTINUOUS] for r in ('direction', 'location', 'shape', 'genes')},
    'needs cutoff or k': [NEEDS[k] for k in CONTINUOUS], 'decision': [decision[k] for k in CONTINUOUS]}, index=CONTINUOUS)
DECISION.to_csv(tables / 'continuous_choice.csv')

fig = plt.figure(figsize=(180 * MM, 150 * MM))
gs = fig.add_gridspec(2, 2, height_ratios=[1.25, 1], width_ratios=[1, 1], hspace=.2, wspace=.15, left=.07, right=.995,
                      top=.97, bottom=.02)
ax = fig.add_subplot(gs[0, 0])
ax.plot([0, expected[0]], [0, expected[0]], color=MUTED, lw=.7, ls='--')
for k in CONTINUOUS:
    pooled = np.sort(np.concatenate([RELABEL_P[k][s] for s in SWAPS]))
    if np.all(pooled >= 1 - 1e-12):
        continue
    observed = -np.log10(pooled)
    ax.plot(expected, observed, color=METHOD_COLOR[k], lw=1.4, ls='--' if not METHOD_ROW[k]['specific'] else '-')
    ax.text(expected[0] + .05, observed[0], f"{NUMBER[k]}: {METHOD_ROW[k]['relabeled calls']}", color=METHOD_COLOR[k],
            fontsize=5.6, va='center')
ax.plot(expected, np.zeros(n_pool), color=METHOD_COLOR['A6·ORAc'], lw=2.2)
ax.plot(expected, np.zeros(n_pool), color=METHOD_COLOR['B2·split'], lw=1, ls=(0, (3, 3)))
ax.annotate('4 and 6: 0, 0. No gene passes the\nsplit-plot test under relabeling,\nso every p = 1 (flat line at 0)',
            xy=(.6, 0), xytext=(.1, .62), textcoords=('data', 'axes fraction'), fontsize=5.3, va='top',
            arrowprops=dict(arrowstyle='-', lw=.5, color=MUTED))
ax.text(.03, .97, 'above the diagonal:\nmore small p than chance', transform=ax.transAxes, fontsize=5.2, color=MUTED, va='top')
ax.set_xlim(0, expected[0] * 1.15)
ax.set_xlabel('expected −log10 p (uniform)')
ax.set_ylabel('observed −log10 p, both relabelings')
ax.text(-.2, 1.02, 'a', fontsize=9, fontweight='bold', transform=ax.transAxes)
crit = fig.add_subplot(gs[0, 1])
crit.axis('off')
crit.text(0, .95, 'Criteria, in order\n\n1  The relabeling rule: NCDR < 0.25 and each\n   relabeling < 5% of tested pathways.\n\n'
          '2  Calibration: at most 0.075 of the pooled\n   relabeled p are ≤ 0.05 (threshold set at\n   this revision, after the '
          'relabeling\n   results were known).\n\n3  The most species calls.\n\n4  What a call returns, and whether a\n'
          '   gene cutoff or k must be chosen.', fontsize=6.2, va='top', color=INK, linespacing=1.25)
tab = fig.add_subplot(gs[1, :])
tab.axis('off')
tab.text(-.02, 1.02, 'b', fontsize=9, fontweight='bold', transform=tab.transAxes)
shown = DECISION.drop(columns=['#']).copy()
shown['method'] = [f'{NUMBER[k]} {SHORT_NAME[k]}' for k in CONTINUOUS]
headers = ['method', '1 relabeled\ncalls (NCDR)', '2 relabeled\np ≤ 0.05', '3 calls', 'dir.', 'loc.', 'shape', 'genes',
           'needs cutoff\nor k', 'decision']
table = tab.table(cellText=shown.astype(str).values, colLabels=headers, loc='upper left', cellLoc='center',
                  colWidths=[.18, .12, .09, .045, .035, .035, .04, .04, .1, .265], bbox=[0, 0, 1, .98])
table.auto_set_font_size(False)
table.set_fontsize(5.4)
for (r, c), cell in table.get_celld().items():
    cell.set_edgecolor('#EEEEEE')
    if r == 0:
        cell.set_text_props(fontweight='bold')
        continue
    k = CONTINUOUS[r - 1]
    if k == CHOSEN:
        cell.set_facecolor('#FFF4CC')
    if c == 0:
        cell.set_text_props(color=METHOD_COLOR[k], fontweight='bold', ha='left')
        cell._loc = 'left'
    if c == len(headers) - 1:
        cell.set_text_props(ha='left')
        cell._loc = 'left'
    if c in (1, 2) and cell.get_text().get_text().startswith('fail'):
        cell.set_text_props(color='#C0392B')
for ext in ('pdf', 'png'):
    fig.savefig(figures / f'fig2_choosing_continuous.{ext}', bbox_inches='tight')
plt.show()
display(DECISION)
print('The rule picks', CHOSEN, f'(method {NUMBER[CHOSEN]}).')
if CHOSEN != 'A6·JNT':
    raise RuntimeError(f'The rule picks {CHOSEN}, not method 3; Part 2 is written for method 3 and must be revised first.')

## 3 · Part 2: the chosen method against the conventional analyses

Method 3, chosen in section 2.6, is compared with three conventional analyses:
- the whole PT (method 1);
- the S1/S2/S3 steps (method 2), which use the same set test as method 3 with position coded as
  three zones;
- S1/S2/S3 analysed as three separate clusters (method 8).

### 3.1 · Method 1: whole PT, pseudobulk DE + signed GSEA (A1·GSEA)

**What it does.**
1. Each specimen's counts are summed over the whole PT.
2. DESeq2 tests every gene, human against mouse.
3. Genes are ranked by their Wald statistic, and preranked GSEA asks whether a pathway's genes
   gather at the top or the bottom of the ranking.

This is the analysis most papers run.

**One call means:** averaged over the whole PT, this pathway's genes are shifted between species
more than other genes are, in the direction given by the sign of NES.

**What it cannot tell you:**
- where along the tubule the difference is;
- whether a difference is real or specimen-to-specimen. With two specimens per species, GSEA's gene
  permutation null ignores that specimens differ, and the relabeling check below shows the cost.

The statistics are notebook 31's, loaded and checked against notebook 37's counts.

In [ ]:
deseq = pd.read_csv(nb31 / 'conventional_deseq2_all_partitions.csv')
gsea31 = pd.read_csv(nb31 / 'conventional_signed_gsea_all_partitions.csv')
whole = gsea31[gsea31.family.eq('whole_PT')]
whole_species = whole[whole.partition.eq('species')].set_index('pathway_id')
register('A1·GSEA', 1, 'Whole PT: pseudobulk DE + signed GSEA', {p: whole.pathway_id[whole.partition.eq(p) & whole.q.le(ALPHA)] for p in PARTS},
         evidence=-np.log10(whole_species.p.clip(lower=1e-300)), returns=('yes/no', 'direction', 'genes'), runtime=LOADED)
check_counts('A1·GSEA', '1 · Whole-PT pseudobulk + signed GSEA')
WHOLE_DIRECTION = np.sign(whole_species.NES)
SPECIES_Q['A1·GSEA'] = whole_species.q.reindex(pathways).fillna(1.).to_numpy(float)

### 3.2 · Method 2: discrete S1/S2/S3 segment steps with the joint test (A4·JNT)

**What it does.**
- For each gene, it asks whether the step pattern from S1 to S2 to S3 differs between species,
  beyond a species offset. The statistic is the partial F of the species × segment terms, with
  shared segment intercepts, computed on the structures.
- It then asks whether a pathway's genes rank higher than genes matched on expression, detection
  and coverage. The z-score is divided by √VIF, which allows for correlation between the pathway's
  genes. This is the **joint test**, the same set test as method 3.
- Position enters only as three zones, so no coordinate is needed.

**One call means:** the pathway's S1 → S2 → S3 profile differs in shape between species more than
matched genes do.

**What it cannot tell you:**
- anything inside a segment;
- an average offset, which the model removes on purpose.

The statistic is computed exactly from notebook 45's cell summaries and checked against notebooks 45
and 50.

In [ ]:
from pseudospace.pathways import build_pathway_membership
from pseudospace.specimen_null import matched_auc_null, summary_partial_f

pool = newest(nb45_cache, 'pool_our_cohort')
assert list(pool['genes'].astype(str)) == list(genes), 'notebook 45 pool genes differ from the universe'
universe12 = pd.read_csv(nb12 / 'gene_statistics.csv', index_col=0).index.tolist()
coverage45 = pd.concat([build_pathway_membership(
    json.loads((data_root / 'mouse_vs_human' / 'pathway_gene_sets' / f'{lib}.json').read_text()), universe12,
    ortholog_map=pd.read_csv(results_root / 'minimal_pt_scfates' / 'ortholog_map_used.csv'), library_name=lib,
    tested=universe12, min_genes=1) for lib in ('Reactome_2022', 'MSigDB_Hallmark_2020', 'KEGG_2019_Mouse')], ignore_index=True)
coverage45['pathway_id'] = coverage45.library + '::' + coverage45.pathway
coverage45 = coverage45[coverage45.n_tested.between(10, min(300, len(universe12) - 1))]
pool_genes = set(pool['genes'].astype(str))
sets45 = {row.pathway_id: [g for g in row.genes_present if g in pool_genes] for row in coverage45.itertuples()}
sets45 = {p: sets45[p] for p in pool['pathways'].astype(str)}   # notebook 45's order and members
pool_vif = pd.Series(pool['vif'], index=list(sets45))
pool_donors = list(pool['donors'].astype(str))
species_of = {d: float(d in SPECIMENS['human']) for d in pool_donors}
designs45 = {'species': (species_of, None)}
designs45.update({s: ({d: float(d in partitions[s]) for d in pool_donors}, species_of) for s in SWAPS})


def step_statistics():
    return {p: summary_partial_f(pool, pool_donors, group, nuisance)['T_steps'] for p, (group, nuisance) in designs45.items()}


t_steps, sec_steps = timed('t_steps', step_statistics, kind='payload', inputs={'pool': digest(pool['mean'])})
step_tests = {p: matched_auc_null(pd.DataFrame({'T_steps': t_steps[p]}, index=genes), sets45, pool['strata']).set_index('pathway_id')
              for p in PARTS}
for p in PARTS:
    step_tests[p]['z_joint'] = step_tests[p].z_matched / np.sqrt(pool_vif)
register('A4·JNT', 2, 'Discrete: S1/S2/S3 segment steps, joint test',
         {p: t.index[t.effect.gt(0) & (bh(norm.sf(t.z_joint)) <= ALPHA)] for p, t in step_tests.items()},
         evidence=step_tests['species'].z_joint, returns=('yes/no', 'location'), runtime=sec_steps)
steps50 = pd.read_csv(nb50 / 'cohort_step_screen_per_pathway.csv').set_index('pathway_id')
assert CALLS['A4·JNT']['species'] == set(steps50.index[steps50.called.astype(bool)])          # Guard: notebook 50
anchor = pd.read_csv(nb45 / 'anchor_our_cohort.csv').set_index('strategy').loc['P1 T_steps joint']
assert [len(CALLS['A4·JNT'][p]) for p in PARTS] == [anchor.species, anchor.relabel_Ctrl1A2, anchor.relabel_Ctrl1A4]   # Guard: notebook 45
SPECIES_Q['A4·JNT'] = bh(norm.sf(step_tests['species'].z_joint.reindex(pathways)))

### 3.3 · Method 8: S1/S2/S3 pseudobulk DE + ORA (A3·ORAc)

**What it does.**
1. Each specimen's counts are summed within S1, S2 and S3.
2. DESeq2 tests every gene within each segment, human against mouse (notebook 31's per-segment
   runs).
3. In each segment, genes with BH ≤ 0.05 and |log2 fold change| ≥ 1 form two lists, human-higher
   and mouse-higher.
4. ORA asks whether a pathway holds more listed genes than chance. One BH correction covers
   1,513 pathways × 3 segments × 2 directions.
5. A pathway is called when any of its six tests passes.

This is the "three clusters" analysis: the segments are treated as separate cell types.

**One call means:** within at least one segment, the pathway's genes are over-represented among
genes that are clearly higher in one species.

**What it cannot tell you:**
- whether the difference changes along the PT. A constant species offset in every segment is
  called as readily as a shape difference;
- anything inside a segment.

It is computed exactly as notebook 63 computes it (from notebook 56's design), and checked against
notebook 56's saved calls. This cell also computes two lists used only by the worked-example rules
of section 4:
- whole-PT ORA, the same test on the whole-PT DESeq2 run;
- S1/S2/S3 GSEA, from notebook 31.

In [ ]:


def deseq_ora(frame, scopes):
    """ORA p per pathway, scope and direction on the DESeq2 lists (BH <= 0.05, |log2FC| >= 1); notebook 63's code."""
    columns, labels = [], []
    for scope in scopes:
        rows = frame[frame.scope.eq(scope)].dropna(subset=['stat']).set_index('gene')
        matrix = membership(gene_sets, list(rows.index))
        for direction in (1, -1):
            selected = rows.padj.le(ALPHA).to_numpy() & (direction * rows.log2FoldChange).ge(LFC).to_numpy()
            columns.append(ora_pvalues(selected, matrix))
            labels.append((scope, direction))
    return pd.DataFrame(np.column_stack(columns), index=pathways, columns=pd.MultiIndex.from_tuples(labels))


def ora_q(pvalues):
    """One BH over every pathway x scope x direction test (notebook 63's ora_summary)."""
    return pd.DataFrame(bh(pvalues.to_numpy().ravel()).reshape(pvalues.shape), index=pvalues.index, columns=pvalues.columns)


SEGMENT_SCOPES = ['PT-S1', 'PT-S2', 'PT-S3']
t0 = time.perf_counter()
seg_p = {p: deseq_ora(deseq[deseq.partition.eq(p)], SEGMENT_SCOPES) for p in PARTS}
seg_q = {p: ora_q(v) for p, v in seg_p.items()}
register('A3·ORAc', 8, 'Conventional: S1/S2/S3 pseudobulk DE + ORA',
         {p: seg_q[p].index[seg_q[p].le(ALPHA).any(axis=1)] for p in PARTS},
         evidence=-np.log10(seg_p['species'].min(axis=1).clip(lower=1e-300)), returns=('yes/no', 'direction', 'location', 'genes'),
         runtime=time.perf_counter() - t0)
saved56 = pd.read_csv(nb56 / 'tables' / 'f_species_calls_by_approach.csv', index_col=0)
assert CALLS['A3·ORAc']['species'] == set(saved56.index[saved56['S1/S2/S3 pseudobulk DE + ORA']])   # Guard: notebook 56
assert [len(CALLS['A3·ORAc'][p]) for p in PARTS] == [118, 0, 1]                                     # Guard: notebook 63
SPECIES_Q['A3·ORAc'] = seg_q['species'].min(axis=1).to_numpy()
SEGMENT_Q = {seg[-2:]: seg_q['species'].xs(seg, axis=1, level=0).min(axis=1).to_numpy() for seg in SEGMENT_SCOPES}

# Lists used only by the worked-example rules (section 4).
whole_p = deseq_ora(deseq[deseq.partition.eq('species')], ['whole_PT'])
RULE_CALLS = {'A1·ORAc': set(whole_p.index[ora_q(whole_p).le(ALPHA).any(axis=1)]), 'A3·ORAc': CALLS['A3·ORAc']['species']}
RULE_EVIDENCE = {'A1·ORAc': -np.log10(whole_p.min(axis=1).clip(lower=1e-300)), 'A3·ORAc': EVIDENCE['A3·ORAc']}
segs = gsea31[gsea31.family.eq('segments') & gsea31.partition.eq('species')]
RULE_CALLS['A3·GSEA'] = set(segs.pathway_id[segs.q.le(ALPHA)])
assert RULE_CALLS['A1·ORAc'] == set(saved56.index[saved56['Whole-PT pseudobulk DE + ORA']])     # Guard: notebook 56
assert len(RULE_CALLS['A3·GSEA']) == spec37.loc['2 · S1/S2/S3 pseudobulk + signed GSEA', 'species_calls']   # Guard: notebook 37

### 3.4 · One table and the specificity of the four methods

The table lists all eight methods. Calibration (criterion 2 of section 2.6) is given for the
continuous methods only.

**The figure.**
- Filled dot: each method's species calls.
- Open dots: the calls under the two specimen relabelings, where no species difference is left to
  find.
- Dashed line: 5% of the tested pathways, the most one relabeling may call.

A specific method has its filled dot far to the right of its open dots.

In [ ]:
METHOD_ROW = {m['key']: m for m in METHODS}
NUMBER = {m['key']: m['#'] for m in METHODS}
IS_CALLED = {k: np.isin(pathways, list(CALLS[k]['species'])) for k in METHOD_ROW}
ORDER = sorted(METHOD_ROW, key=lambda k: NUMBER[k])
GROUP = {k: 'continuous' if k in CONTINUOUS else 'conventional reference' for k in ORDER}
NEEDS.update({'A1·GSEA': 'no', 'A4·JNT': 'no', 'A3·ORAc': 'gene cutoff'})
TABLE = pd.DataFrame(METHODS).set_index('key').loc[ORDER]
TABLE.insert(1, 'group', [GROUP[k] for k in ORDER])
TABLE['relabeling rule'] = np.where(TABLE.specific, 'pass', 'fail')
TABLE['share of relabeled p ≤ 0.05'] = [CALIBRATION.get(k, np.nan) for k in ORDER]
TABLE['returns'] = [', '.join(r for r in RETURNS if TABLE.loc[k, r]) for k in ORDER]
TABLE['needs cutoff or k'] = [NEEDS[k] for k in ORDER]
TABLE['chosen'] = [k == CHOSEN for k in ORDER]
TABLE.to_csv(tables / 'methods.csv')
display(TABLE[['#', 'group', 'method', 'species calls', 'relabeled calls', 'NCDR', 'relabeling rule',
               'share of relabeled p ≤ 0.05', 'returns', 'needs cutoff or k']].round(3))

PART2 = [CHOSEN, 'A1·GSEA', 'A4·JNT', 'A3·ORAc']
fig, ax = plt.subplots(figsize=(120 * MM, 36 * MM))
y = np.arange(len(PART2))[::-1]
for yi, key in zip(y, PART2):
    species_n = len(CALLS[key]['species'])
    relabeled = [len(CALLS[key][s]) for s in SWAPS]
    ax.plot([min([species_n] + relabeled) + .5, max([species_n] + relabeled) + .5], [yi, yi], color=LIGHT, lw=1, zorder=1)
    ax.scatter([r + .5 for r in relabeled], [yi, yi], s=22, facecolor='white', edgecolor=NULL, lw=1, zorder=2)
    ax.scatter(species_n + .5, yi, s=26, color=METHOD_COLOR[key], zorder=3)
    ax.text(1250, yi, f"{species_n} vs {', '.join(map(str, relabeled))} · {'pass' if TABLE.loc[key, 'specific'] else 'fail'}",
            va='center', fontsize=5.6, color=INK if TABLE.loc[key, 'specific'] else MUTED)
ax.axvline(NULL_RATE * len(pathways) + .5, color=NULL, lw=.7, ls='--')
ax.set_xscale('log')
ax.set_xlim(.4, 1200)
ax.set_xticks([.5, 1.5, 10.5, 100.5, 1000.5], ['0', '1', '10', '100', '1000'])
ax.xaxis.set_minor_formatter(plt.NullFormatter())
ax.set_yticks(y, [f'{NUMBER[k]} {SHORT_NAME[k]}' for k in PART2])
for tick, key in zip(ax.get_yticklabels(), PART2):
    tick.set_color(METHOD_COLOR[key])
ax.set_xlabel('pathways called (log scale)')
ax.scatter([], [], s=26, color=INK, label='species split')
ax.scatter([], [], s=22, facecolor='white', edgecolor=NULL, lw=1, label='specimen relabelings')
ax.plot([], [], color=NULL, lw=.7, ls='--', label='5% of tested pathways')
ax.legend(loc='upper left', bbox_to_anchor=(0, 1.35), ncol=3)
for ext in ('pdf', 'png'):
    fig.savefig(figures / f'fig5_specificity_vs_conventional.{ext}', bbox_inches='tight')
plt.show()

### 3.5 · The chosen method against each conventional analysis, pathway by pathway

Each dot is a pathway: −log10 q of a conventional analysis (x) against the chosen method, method 3
(y). Dashed lines mark q = 0.05; the corner counts give the four quadrants.
- **a. Method 2, the S1/S2/S3 steps.** The pathways called by only one of the two are named. Names
  in red are called by no specific discrete method: neither method 2 nor method 8.
- **b. Method 8, any segment.**
- **c. Method 1, the whole PT.**
- **d–f. Method 8 in S1, S2 and S3 separately.** The q of a segment is the smaller of its
  human-higher and mouse-higher tests.

**Offset-leaning pathways (triangles).** A pathway leans towards offset when its genes stand out more
on the species offset (T_level) than on the positional shape (T_spatial). Both are notebook 37's
matched rank z for the species split.

This is a relative label, and a weak one:
- about 45% of all pathways lean towards offset by it;
- 66 of the 92 calls that only method 8 makes lean towards offset.

Both numbers are computed below. Method 8 therefore finds offset-leaning pathways more often than
chance, but not only those.

In [ ]:
level37 = tests37[tests37.partition.eq('species') & tests37.statistic.eq('T_level')].set_index('pathway_id').reindex(pathways)
z_offset, z_shape = level37.z_matched.to_numpy(float), by_part['species'].z_matched.to_numpy(float)
OFFSET_LEANING = z_offset > z_shape
only8 = IS_CALLED['A3·ORAc'] & ~IS_CALLED[CHOSEN]
print(f'Offset-leaning: {OFFSET_LEANING.mean():.0%} of all pathways; {int((only8 & OFFSET_LEANING).sum())} of the '
      f'{int(only8.sum())} calls made only by method 8; {int((IS_CALLED[CHOSEN] & OFFSET_LEANING).sum())} of method 3\'s '
      f'{int(IS_CALLED[CHOSEN].sum())} calls.')
continuous_only = sorted(CALLS[CHOSEN]['species'] - CALLS['A4·JNT']['species'])
steps_only = sorted(CALLS['A4·JNT']['species'] - CALLS[CHOSEN]['species'])
no_discrete = [p for p in continuous_only if p not in CALLS['A3·ORAc']['species']]
scatter_rng = np.random.default_rng(12)


def axis_values(q):
    v = -np.log10(np.asarray(q, float))
    out = v.copy()
    pile = v < -np.log10(.99)
    out[pile] = scatter_rng.uniform(*BAND, pile.sum())
    return out


def versus(ax, qx, called_x, color_x, label_x, size=5, count_size=5.2, highlight=None):
    """Scatter of a conventional method's −log10 q (x) against the chosen method's (y), with quadrant counts."""
    x, y = axis_values(qx), axis_values(SPECIES_Q[CHOSEN])
    a, b = called_x, IS_CALLED[CHOSEN]
    colors = np.where(a & b, INK, np.where(a, color_x, np.where(b, METHOD_COLOR[CHOSEN], LIGHT)))
    order = np.argsort((a | b).astype(int), kind='stable')
    for marker, flag in (('o', ~OFFSET_LEANING), ('^', OFFSET_LEANING)):
        sel = order[flag[order]]
        ax.scatter(x[sel], y[sel], s=size, c=colors[sel], marker=marker, linewidths=0, zorder=2)
    if highlight is not None:
        ax.scatter(x[highlight], y[highlight], s=size * 3.5, facecolor='none', edgecolor='#C0392B', lw=.8, zorder=3)
    ax.axvline(THR, color=MUTED, lw=.6, ls='--', zorder=1)
    ax.axhline(THR, color=MUTED, lw=.6, ls='--', zorder=1)
    ax.axvspan(BAND[0] - .03, BAND[1] + .03, color='#EEEEEE', lw=0, zorder=0)
    ax.axhspan(BAND[0] - .03, BAND[1] + .03, color='#EEEEEE', lw=0, zorder=0)
    ax.set_xlim(BAND[0] - .08, max(x.max(), THR) * 1.06)
    ax.set_ylim(BAND[0] - .08, max(y.max(), THR) * 1.06)
    counts = {'both': int((a & b).sum()), 'conventional only': int((a & ~b).sum()), 'method 3 only': int((~a & b).sum()),
              'neither': int((~a & ~b).sum())}
    box = dict(boxstyle='round,pad=.15', facecolor='white', edgecolor='none', alpha=.85)
    ax.text(.98, .98, f"both {counts['both']}", ha='right', va='top', fontsize=count_size, transform=ax.transAxes, bbox=box)
    ax.text(.02, .98, f"3 only {counts['method 3 only']}", ha='left', va='top', fontsize=count_size, transform=ax.transAxes, bbox=box)
    ax.text(.98, .12, f"{label_x} only {counts['conventional only']}", ha='right', va='bottom', fontsize=count_size,
            transform=ax.transAxes, bbox=box)
    ax.text(.98, .02, f"neither {counts['neither']}", ha='right', va='bottom', fontsize=count_size, color=MUTED,
            transform=ax.transAxes, bbox=box)
    return counts


QUADRANTS = {}
fig = plt.figure(figsize=(180 * MM, 175 * MM))
gs = fig.add_gridspec(3, 3, height_ratios=[1.5, 1, 1], hspace=.38, wspace=.32, left=.07, right=.98, top=.95, bottom=.05)
ax = fig.add_subplot(gs[0, :])
in_none = np.isin(pathways, no_discrete)
QUADRANTS['2 S1/S2/S3 steps'] = versus(ax, SPECIES_Q['A4·JNT'], IS_CALLED['A4·JNT'], METHOD_COLOR['A4·JNT'], '2', size=9,
                                       count_size=6, highlight=in_none)
x_steps, y_chosen = -np.log10(pd.Series(SPECIES_Q['A4·JNT'], index=pathways)), -np.log10(pd.Series(SPECIES_Q[CHOSEN], index=pathways))
label_y = np.linspace(THR + .1, y_chosen.max() + .05, len(continuous_only))
for p, yt in zip(sorted(continuous_only, key=lambda p: y_chosen[p]), label_y):
    ax.annotate(NAMES[p][:46], xy=(x_steps[p], y_chosen[p]), xytext=(.02, yt), fontsize=5.2, va='center',
                color='#C0392B' if p in no_discrete else INK, arrowprops=dict(arrowstyle='-', lw=.35, color=LIGHT, shrinkB=2))
for k, p in enumerate(steps_only):
    ax.annotate(NAMES[p], xy=(x_steps[p], y_chosen[p]), xytext=(1.75, .95 - .17 * k), fontsize=5.2, va='center',
                arrowprops=dict(arrowstyle='-', lw=.35, color=LIGHT, shrinkB=2))
ax.scatter([], [], s=12, c=INK, marker='o', label='shape-leaning')
ax.scatter([], [], s=12, c=INK, marker='^', label='offset-leaning')
ax.scatter([], [], s=30, facecolor='none', edgecolor='#C0392B', lw=.8, label='called by no specific discrete method (red names)')
ax.legend(loc='lower center', bbox_to_anchor=(.5, 1.0), ncol=3)
ax.set_xlabel('2 S1/S2/S3 steps  −log10 q', color=METHOD_COLOR['A4·JNT'])
ax.set_ylabel('3 joint test  −log10 q', color=METHOD_COLOR[CHOSEN])
ax.text(-.06, 1.02, 'a', fontsize=9, fontweight='bold', transform=ax.transAxes)
panels = [('b', gs[1, 0], SPECIES_Q['A3·ORAc'], IS_CALLED['A3·ORAc'], 'A3·ORAc', '8', '8 S1/S2/S3 ORA, any segment'),
          ('c', gs[1, 1], SPECIES_Q['A1·GSEA'], IS_CALLED['A1·GSEA'], 'A1·GSEA', '1', '1 whole PT, GSEA')]
panels += [(letter, gs[2, j], SEGMENT_Q[seg], SEGMENT_Q[seg] <= ALPHA, 'A3·ORAc', '8', f'8 S1/S2/S3 ORA, {seg}')
           for j, (letter, seg) in enumerate((('d', 'S1'), ('e', 'S2'), ('f', 'S3')))]
for letter, cell, q, called, key, short, label in panels:
    ax = fig.add_subplot(cell)
    QUADRANTS[label] = versus(ax, q, called, METHOD_COLOR[key], short)
    ax.set_xlabel(f'{label}  −log10 q', color=METHOD_COLOR[key])
    ax.set_ylabel('3  −log10 q', color=METHOD_COLOR[CHOSEN])
    ax.text(-.22, 1.04, letter, fontsize=9, fontweight='bold', transform=ax.transAxes)
note = fig.add_subplot(gs[1, 2])
note.axis('off')
note.text(0, .95, 'Black: called by both. Colour: called by one\nmethod only. Grey: neither. Grey bands:\nq ≥ 0.99, jittered.\n\n'
          'Triangles: offset-leaning pathways\n(T_level matched z > T_spatial matched z).', fontsize=5.6, color=MUTED, va='top')
for ext in ('pdf', 'png'):
    fig.savefig(figures / f'fig6_chosen_vs_conventional.{ext}', bbox_inches='tight')
plt.show()
display(pd.DataFrame(QUADRANTS).T)

### 3.6 · What each side calls that the other does not

**Method 3 against method 2.** These two use the same set test and differ only in how position
enters. The first table lists the pathways method 3 calls and method 2 does not. For each:
- **step-screen q:** method 2's q-value, showing how close the discrete screen came;
- **called by method 8:** whether the per-segment method calls it. If not, no specific discrete
  method finds the pathway;
- **probe-sensitive:** method 3's call fails notebook 37's equal-probe check. Restricted to genes
  with equal probe counts in both species, the pathway keeps less than 70% of its effect or q
  exceeds 0.10.

The second table lists the pathways method 2 calls and method 3 does not.

**Method 8 against method 3.** The third table splits the pathways method 8 calls and method 3 does
not into offset-leaning and shape-leaning (section 3.5), with three examples each, the smallest
method 8 q first.

In [ ]:
probe_pass = table37.probe_equal_pass.reindex(pathways)
q3 = pd.Series(SPECIES_Q[CHOSEN], index=pathways)
q8 = pd.Series(SPECIES_Q['A3·ORAc'], index=pathways)
ADDED = pd.DataFrame({'pathway': NAMES[continuous_only], 'method 3 q': q3[continuous_only],
                      'step-screen q (method 2)': steps50.q_joint.reindex(continuous_only),
                      'called by method 8': [p in CALLS['A3·ORAc']['species'] for p in continuous_only],
                      'probe-sensitive': ~probe_pass.reindex(continuous_only).astype(bool)}).sort_values('step-screen q (method 2)')
ADDED.to_csv(tables / 'continuous_adds_over_discrete.csv')
STEPS_ONLY = pd.DataFrame({'pathway': NAMES[steps_only], 'step-screen q (method 2)': steps50.q_joint.reindex(steps_only),
                           'method 3 q': q3[steps_only]})
STEPS_ONLY.to_csv(tables / 'discrete_adds_over_continuous.csv')
no8 = ADDED[~ADDED['called by method 8']]
print(f"Method 3 calls {len(continuous_only)} pathways that method 2 does not (step-screen q {ADDED['step-screen q (method 2)'].min():.3f}–"
      f"{ADDED['step-screen q (method 2)'].max():.3f}); {len(no8)} of them are not called by method 8 either "
      f"({int(no8['probe-sensitive'].sum())} probe-sensitive).")
display(ADDED.round(3))
print(f'Method 2 calls {len(steps_only)} pathways that method 3 does not:')
display(STEPS_ONLY.round(3))
only8_ids = np.array(pathways)[only8]
rows8 = []
for label, flag in (('offset-leaning', True), ('shape-leaning', False)):
    ids = [p for p in only8_ids if OFFSET_LEANING[pathways.index(p)] == flag]
    rows8.append({'lean': label, 'pathways': len(ids), 'median method 3 q': float(q3[ids].median()) if ids else np.nan,
                  'method 3 q ≤ 0.25': int((q3[ids] <= .25).sum()), 'called by method 2': sum(p in CALLS['A4·JNT']['species'] for p in ids),
                  'examples': '; '.join(NAMES[q8[ids].sort_values(kind='stable').index[:3]])})
EIGHT_ONLY = pd.DataFrame(rows8).set_index('lean')
EIGHT_ONLY.to_csv(tables / 'method8_adds_over_continuous.csv')
display(EIGHT_ONLY.round(3))

## 4 · Worked examples: one pathway, three views

Each row shows one pathway three ways. The value plotted is a display score only, the mean of the
members' species-balanced z-scored expression in each structure; no method above tests it:
- **whole PT:** the specimen means;
- **S1/S2/S3:** specimen × segment means, with bars for the species mean;
- **continuous:** a specimen-balanced species curve (6-df spline), with specimen × 12-bin means as
  points.

The five pathways were chosen by rules fixed in `plan.md` before notebook 61 first ran:
- **E1:** the whole-PT, segment and continuous methods agree. Here it is called by every method
  except method 6.
- **E2:** segment means are flat, but the curves have structure.
- **E3:** a discrete call that has no continuous call.
- **E4:** the direction depends on position.
- **E5:** a step-like pathway, where the curve adds nothing.

Some rules refer to S1/S2/S3 ORA (method 8), whole-PT ORA and S1/S2/S3 GSEA (notebook 63). The last two
lists are computed in section 3.3 only to apply these rules. Each panel title lists which of the
eight methods call the pathway.

In [ ]:
member_index = {p: pd.Index(genes).get_indexer(v) for p, v in gene_sets.items()}
pathway_delta = pd.DataFrame({p: curves['species']['delta'][idx].mean(axis=0) for p, idx in member_index.items()},
                             index=np.round(grid, 4)).T
evidence = pd.DataFrame({**EVIDENCE, **RULE_EVIDENCE})
C = CALLS['A6·JNT']['species']
scores = newest(nb56 / 'stage_cache', 'structure_scores')

beyond = pd.read_csv(nb57 / 'per_pathway_beyond_steps.csv', index_col=0)
ranks = evidence.rank(pct=True)
rules = {}
agree = sorted(RULE_CALLS['A1·ORAc'] & RULE_CALLS['A3·ORAc'] & CALLS['A4·JNT']['species'] & C)
rules['E1 · all methods agree'] = ranks.loc[agree, ['A1·ORAc', 'A3·ORAc', 'A4·JNT', 'A6·JNT']].min(axis=1).sort_values(ascending=False).index[:1] if agree else []
flat = [p for p in C if steps50.q_joint.get(p, 1) > .05 and p not in RULE_CALLS['A3·GSEA'] and p not in RULE_CALLS['A3·ORAc']
        and bool(beyond.loc[p, 'called test_B_coordinate_steps'])]
rules['E2 · segment means flat, curves structured'] = beyond.loc[flat, 'z_joint test_B_coordinate_steps'].sort_values(ascending=False).index[:1] if flat else []
discrete_only = sorted(RULE_CALLS['A3·ORAc'] - C)
rules['E3 · discrete call, no continuous call'] = RULE_EVIDENCE['A3·ORAc'][discrete_only].sort_values(ascending=False).index[:1] if discrete_only else []
reversing = []
for p in C:
    d = pathway_delta.loc[p].to_numpy()
    threshold = .25 * np.abs(d).max()
    if (d >= threshold).any() and (d <= -threshold).any():
        reversing.append(p)
rules['E4 · direction depends on position'] = EVIDENCE['A6·JNT'][reversing].sort_values(ascending=False).index[:1] if reversing else []
steplike = [p for p in C & CALLS['A4·JNT']['species'] if not bool(beyond.loc[p, 'called test_B_coordinate_steps'])]
rules['E5 · step-like: continuous adds nothing'] = steps50.z_joint[steplike].sort_values(ascending=False).index[:1] if steplike else []
candidates = {'E1': len(agree), 'E2': len(flat), 'E3': len(discrete_only), 'E4': len(reversing), 'E5': len(steplike)}
EXAMPLES = {}
for rule, picked in rules.items():
    if len(picked):
        EXAMPLES[rule] = picked[0]
        print(f'{rule}: {NAMES[picked[0]]} (pathways meeting the rule: {candidates[rule[:2]]})')
    else:
        print(f'{rule}: no pathway meets the rule')

In [ ]:
score_matrix = scores['mean_z']
weights = np.zeros(len(position))
for sp, names_ in SPECIMENS.items():
    for n in names_:
        rows = specimen == n
        weights[rows] = len(position) / (2 * len(names_) * rows.sum())
fine_bins = np.clip(((position - lo) / (hi - lo) * EXAMPLE_BINS).astype(int), 0, EXAMPLE_BINS - 1)
transitions = pd.read_csv(nb37 / 'tables' / 'segment_transitions_by_specimen.csv').groupby('transition').position.mean()


def species_curve(values, species_mask):
    design = make_gam_design(position[species_mask], knots)
    w = np.sqrt(weights[species_mask])
    beta, *_ = np.linalg.lstsq(w[:, None] * design, w * values[species_mask], rcond=None)
    return make_gam_design(grid, knots) @ beta


fig, axes = plt.subplots(len(EXAMPLES), 3, figsize=(180 * MM, 38 * MM * len(EXAMPLES)),
                         gridspec_kw={'width_ratios': [.55, 1, 1.8]}, squeeze=False)
example_rows = []
for r, (rule, pathway) in enumerate(EXAMPLES.items()):
    values = score_matrix[:, pathways.index(pathway)]
    ax_w, ax_s, ax_c = axes[r]
    for k, (sp, color) in enumerate((('mouse', MOUSE), ('human', HUMAN))):
        means = [values[specimen == n].mean() for n in SPECIMENS[sp]]
        ax_w.bar(k, np.mean(means), color=color, alpha=.35, width=.6)
        for n, m in zip(SPECIMENS[sp], means):
            ax_w.scatter(k, m, color=color, marker=SPECIMEN_MARKER[n], s=10, zorder=3, edgecolor='white', linewidth=.4)
        for j, seg in enumerate(('PT-S1', 'PT-S2', 'PT-S3')):
            seg_means = [values[(specimen == n) & (segment == seg)].mean() for n in SPECIMENS[sp]]
            xpos = j + (k - .5) * .38
            ax_s.bar(xpos, np.mean(seg_means), width=.36, color=color, alpha=.35)
            for n, m in zip(SPECIMENS[sp], seg_means):
                ax_s.scatter(xpos, m, color=color, marker=SPECIMEN_MARKER[n], s=10, zorder=3, edgecolor='white', linewidth=.4)
        mask = ~human if sp == 'mouse' else human
        ax_c.plot(grid, species_curve(values, mask), color=color, lw=1.4)
        for n in SPECIMENS[sp]:
            rows = specimen == n
            b = pd.Series(values[rows]).groupby(fine_bins[rows]).mean()
            centres = lo + (b.index.to_numpy() + .5) * (hi - lo) / EXAMPLE_BINS
            ax_c.scatter(centres, b.to_numpy(), color=color, marker=SPECIMEN_MARKER[n], s=7, alpha=.8,
                         edgecolor='white', linewidth=.3, zorder=3)
    for t in transitions:
        ax_c.axvline(t, color=NULL, lw=.6, ls=':')
    ax_w.set_xticks([0, 1], ['mouse', 'human'])
    ax_s.set_xticks([0, 1, 2], ['S1', 'S2', 'S3'])
    ax_c.set_xlim(lo, hi)
    ax_w.set_ylabel('display score (members\' mean z)')
    calls_here = [str(n) for n in sorted(int(TABLE.loc[k, '#']) for k in ORDER if pathway in CALLS[k]['species'])]
    ax_w.set_title(f'{rule.split(" · ")[0]}  whole PT', loc='left')
    ax_s.set_title('S1 / S2 / S3', loc='left')
    ax_c.set_title(f'{NAMES[pathway][:60]} · called by methods {", ".join(calls_here) or "none"}', loc='left')
    if r == len(EXAMPLES) - 1:
        ax_c.set_xlabel('PT coordinate (SCF13); dotted lines: mean S1→S2 and S2→S3 transitions')
    example_rows.append({'example': rule, 'pathway': NAMES[pathway], 'called by methods': ', '.join(calls_here),
                         'step-screen q (A4·JNT)': steps50.q_joint.get(pathway, np.nan),
                         'A6·JNT q_joint': tests37[tests37.statistic.eq('T_spatial') & tests37.partition.eq('species')
                                               & tests37.pathway_id.eq(pathway)].q_joint.iloc[0],
                         'beyond coordinate steps called (notebook 57)': bool(beyond.loc[pathway, 'called test_B_coordinate_steps'])})
handles = [plt.Line2D([], [], color=MOUSE, lw=1.4, label='mouse'), plt.Line2D([], [], color=HUMAN, lw=1.4, label='human'),
           plt.Line2D([], [], color=INK, marker='o', lw=0, markersize=3, label='specimen 1'),
           plt.Line2D([], [], color=INK, marker='s', lw=0, markersize=3, label='specimen 2')]
axes[0, 2].legend(handles=handles, loc='upper left', ncol=4, bbox_to_anchor=(0, 1.32))
fig.tight_layout()
for ext in ('pdf', 'png'):
    fig.savefig(figures / f'fig3_worked_examples.{ext}', bbox_inches='tight')
plt.show()
EXAMPLE_TABLE = pd.DataFrame(example_rows)
EXAMPLE_TABLE.to_csv(tables / 'worked_examples.csv', index=False)
display(EXAMPLE_TABLE.round(4))

## 5 · Gene heatmaps: which genes drive the signal, and where

A pathway score averages its members, so it hides the genes that drive a call and cancels members
that change in opposite directions. These heatmaps show the genes themselves.

- **Rows.** At most 40 genes, chosen by T_spatial. A bullet marks genes in the top 10% by
  T_spatial.
- **Left matrix.** Which member pathway, numbered in the title, contains each gene.
- **Columns.** The 12 coordinate bins, mouse and human side by side.
  - Each gene's specimen-averaged bin means are z-scored within each species, which removes the
    level offset.
  - Dotted lines mark each species' mean S1→S2 and S2→S3 transitions; a dot marks the peak bin.
- **Row order.** By mouse peak bin, then human peak bin.
- **Right of the heatmap.**
  - The notebook 43 zonation class.
  - log10 T_spatial.
  - The whole-PT DESeq2 log2 human/mouse (H/M), so the offset is not lost.
- **Below.** Each specimen's own z-scored bins in the same order, so the pattern can be checked
  specimen by specimen.

**Three heatmaps:**
1. **E1, where the methods agree.** The program of method 3's calls that contains E1.
2. **E4, where the continuous method shows a sign change the whole-PT method misses.** The
   program that contains E4. Method 1 does not call it.

A **program** groups method 3's calls by the genes that carry their signal (notebook 37's rule,
made order-invariant). A pathway's driver genes are its members in the top 10% by T_spatial, or all
members when fewer than three qualify. Pathways are joined by average linkage on 1 − overlap
coefficient of driver genes, cut at 0.5. Near-duplicate library entries are therefore shown once.
3. **The gene-first S3 module: method 6's module 3.** This is the curve module of split-plot
   significant genes whose species difference falls toward S3, so mouse is higher there. Its 40
   highest-T_spatial genes are shown, and the left matrix gives its enriched pathways (at most five).
   The species-blind co-expression screen of notebook 63 finds an overlapping 46-gene S3 module that
   shares 31 genes with this one.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap
from pseudospace.pathway_decoys import overlap_programs

PROGRAM_CUT = .5
contributing = set(gene37.index[gene37.ge(gene37.quantile(1 - TOP_SHARE))])


def driver_programs(ids):
    """Order-invariant programs: notebook 37's rule with pathways sorted by pathway_id before clustering."""
    ids = sorted(set(ids))
    if not ids:
        return pd.Series(dtype=int)
    return overlap_programs(gene_sets, ids, contributing=contributing, cut=PROGRAM_CUT).set_index('pathway_id').program


def program_names(labels, score):
    """Name each program after its member with the highest score (ties by pathway id)."""
    names = {}
    for program, block in labels.groupby(labels):
        best = score.reindex(block.index).fillna(-np.inf).sort_index().sort_values(ascending=False, kind='stable').index[0]
        names[program] = NAMES[best]
    return labels.map(names)


c1_drivers = driver_programs(CALLS['A6·JNT']['species'])

MAX_ROWS, Z_CLIP = 40, 2.5
HEAT = LinearSegmentedColormap.from_list('teal_grey_brown', ['#01665E', '#5AB4AC', '#E6E6E6', '#D8B365', '#8C510A'])
CLASS_COLORS = {'conserved': '#009E73', 'reversal': '#CC79A7', 'mouse-only': MOUSE, 'human-only': HUMAN,
                'indeterminate': '#E3E3E3', 'neither': '#B8B8B8'}
SHORT_SPECIMEN = {'Ctrl1A2': 'M1', 'Ctrl1A4': 'M2', 'HUK1_COR1': 'H1', 'HUK1_MED1': 'H2'}


def specimen_bin_means():
    """Mean log-normalised expression per specimen x coordinate bin (genes x bins per specimen)."""
    out = {}
    for n in ALL_SPECIMENS:
        rows = np.flatnonzero(specimen == n)
        indicator = sparse.csr_matrix((np.ones(len(rows)), (fine_bins[rows], np.arange(len(rows)))),
                                      shape=(EXAMPLE_BINS, len(rows)))
        sums = np.asarray((indicator @ inputs['Y'][rows]).todense())
        out[n] = (sums / np.asarray(indicator.sum(axis=1))).T
    return out


bin_means = cached_payload('specimen_bin_means', specimen_bin_means, root=cache, params={'bins': EXAMPLE_BINS},
                           inputs=input_key, code=code)
species_means = {sp: np.mean([bin_means[n] for n in SPECIMENS[sp]], axis=0) for sp in SPECIMENS}


def zscore_rows(values):
    centred = values - values.mean(axis=1, keepdims=True)
    sd = values.std(axis=1, keepdims=True)
    return np.where(sd > 1e-9, centred / np.where(sd > 1e-9, sd, 1), 0.)


species_z = {sp: zscore_rows(v) for sp, v in species_means.items()}
specimen_z = {n: zscore_rows(v) for n, v in bin_means.items()}
gene_position = pd.Series(np.arange(len(genes)), index=genes)
t_spatial = gene37
top_decile = set(t_spatial.nlargest(int(round(TOP_SHARE * len(genes)))).index)
whole_lfc = deseq[deseq.partition.eq('species') & deseq.scope.eq('whole_PT')].set_index('gene').log2FoldChange
zonation = pd.read_csv(results_root / 'pt_revision_classes' / 'reviewed' / 'tables' / 'gene_classes_symmetric.csv',
                       index_col=0)['union S2-S1 + S3c-early']
species_transitions = pd.read_csv(nb37 / 'tables' / 'segment_transitions_by_specimen.csv').groupby(
    ['species', 'transition']).position.mean()
bin_edges = np.linspace(lo, hi, EXAMPLE_BINS + 1)


def program_rows(member_pathways):
    """Union of the members' contributing genes (fallback: all members), capped at the top 40 by T_spatial."""
    union = []
    for p in member_pathways:
        members = [g for g in gene_sets[p] if g in gene_position.index]
        chosen = [g for g in members if g in contributing]
        union.extend(chosen if len(chosen) >= 3 else members)
    union = sorted(set(union))
    shown = union if len(union) <= MAX_ROWS else list(t_spatial[union].nlargest(MAX_ROWS).index)
    idx = gene_position[shown].to_numpy()
    peaks = pd.DataFrame({'mouse': species_z['mouse'][idx].argmax(axis=1), 'human': species_z['human'][idx].argmax(axis=1),
                          'gene': shown})
    order = peaks.sort_values(['mouse', 'human', 'gene']).index.to_numpy()
    return [shown[i] for i in order], len(union)


def draw_program_heatmap(member_pathways, title, rows=None):
    """Gene heatmap of a program (rows from its members' driver genes) or of given genes (``rows``, e.g. a module)."""
    member_pathways = sorted(member_pathways, key=lambda p: -EVIDENCE['A6·JNT'].get(p, 0))
    if rows is None:
        rows, n_union = program_rows(member_pathways)
    else:
        n_union = len(rows)
    idx = gene_position[rows].to_numpy()
    n, k = len(rows), len(member_pathways)
    height_mm = 40 + n * 2.4 * 1.6 + 3 * k
    fig = plt.figure(figsize=(180 * MM, height_mm * MM))
    widths = [max(k, 1) * .9, 12, 1.2, 12, 1.2, 2.4, 2.4]
    outer = fig.add_gridspec(2, 1, height_ratios=[1, .6], hspace=.16, top=1 - (14 + 3 * k) / height_mm, bottom=.04,
                             left=.12, right=.98)
    top = outer[0].subgridspec(1, 7, width_ratios=widths, wspace=.06)
    bottom = outer[1].subgridspec(1, 7, width_ratios=widths, wspace=.06)
    mx = fig.add_subplot(top[0, 0])
    matrix = np.array([[g in set(gene_sets[p]) for p in member_pathways] for g in rows], float)
    yy, xx = np.nonzero(matrix)
    mx.scatter(xx, yy, s=4, color=INK, linewidths=0)
    mx.set_xlim(-.5, max(k, 1) - .5)
    mx.set_ylim(n - .5, -.5)
    mx.set_xticks(range(k), [str(i + 1) for i in range(k)], fontsize=5)
    mx.xaxis.tick_top()
    mx.set_yticks(range(n), [('• ' if g in top_decile else '') + g for g in rows], fontsize=5)
    for side in ('right', 'bottom'):
        mx.spines[side].set_visible(False)
    mx.set_title('in pathway' if k else '', loc='left', fontsize=5.5, pad=8)
    for col, sp in ((1, 'mouse'), (3, 'human')):
        ax = fig.add_subplot(top[0, col])
        image = ax.imshow(np.clip(species_z[sp][idx], -Z_CLIP, Z_CLIP), cmap=HEAT, vmin=-Z_CLIP, vmax=Z_CLIP,
                          aspect='auto', interpolation='nearest')
        ax.scatter(species_z[sp][idx].argmax(axis=1), np.arange(n), s=3, color=INK, zorder=3, linewidths=0)
        for t in ('S1→S2', 'S2→S3'):
            ax.axvline((species_transitions[(sp, t)] - lo) / (hi - lo) * EXAMPLE_BINS - .5, color=INK, lw=.6, ls=':')
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_title(f'{sp}: specimen-averaged, z within species', loc='left', color=MOUSE if sp == 'mouse' else HUMAN)
        sub = bottom[0, col].subgridspec(1, 2, wspace=.08)
        for j, name in enumerate(SPECIMENS[sp]):
            bx = fig.add_subplot(sub[0, j])
            bx.imshow(np.clip(specimen_z[name][idx], -Z_CLIP, Z_CLIP), cmap=HEAT, vmin=-Z_CLIP, vmax=Z_CLIP,
                      aspect='auto', interpolation='nearest')
            for t in ('S1→S2', 'S2→S3'):
                bx.axvline((species_transitions[(sp, t)] - lo) / (hi - lo) * EXAMPLE_BINS - .5, color=INK, lw=.5, ls=':')
            bx.set_xticks([0, EXAMPLE_BINS - 1], ['start', 'end'], fontsize=5)
            bx.set_yticks([])
            bx.set_title(f'{SHORT_SPECIMEN[name]} ({name})', loc='left', fontsize=5.5, fontweight='normal')
    cx = fig.add_subplot(top[0, 4])
    for i, g in enumerate(rows):
        c = zonation.get(g, np.nan)
        cx.add_patch(plt.Rectangle((0, i - .4), 1, .8, color=CLASS_COLORS.get(c, '#FFFFFF'),
                                   ec=LIGHT if c not in CLASS_COLORS else 'none', lw=.3))
    cx.set_xlim(0, 1)
    cx.set_ylim(n - .5, -.5)
    cx.axis('off')
    cx.set_title('class', loc='left', fontsize=5.5)
    tx = fig.add_subplot(top[0, 5])
    tx.barh(np.arange(n), np.log10(t_spatial[rows].clip(lower=1)), color=ACCENT, height=.7)
    tx.set_ylim(n - .5, -.5)
    tx.set_yticks([])
    tx.set_title('log10 T', loc='left', fontsize=5.5)
    tx.tick_params(labelsize=5)
    lx = fig.add_subplot(top[0, 6])
    lfc = whole_lfc.reindex(rows).clip(-6, 6).fillna(0).to_numpy()
    lx.barh(np.arange(n), lfc, color=np.where(lfc > 0, HUMAN, MOUSE), height=.7)
    lx.axvline(0, color=INK, lw=.5)
    lx.set_xlim(-6.5, 6.5)
    lx.set_ylim(n - .5, -.5)
    lx.set_yticks([])
    lx.set_title('log2 H/M', loc='left', fontsize=5.5)
    lx.tick_params(labelsize=5)
    cax = fig.add_subplot(bottom[0, 4:])
    cax.axis('off')
    bar = fig.colorbar(image, ax=cax, orientation='horizontal', fraction=.45, aspect=10, pad=0)
    bar.set_label('z within species (clipped ±2.5)', fontsize=5)
    bar.ax.tick_params(labelsize=5)
    legend = [plt.Rectangle((0, 0), 1, 1, color=v) for v in CLASS_COLORS.values()] + \
        [plt.Rectangle((0, 0), 1, 1, facecolor='white', edgecolor=LIGHT)]
    cax.legend(legend, list(CLASS_COLORS) + ['not classified'], loc='upper left', fontsize=4.5, ncol=2,
               handlelength=1, borderaxespad=0)
    called = [str(n) for n in sorted(int(TABLE.loc[m, '#']) for m in ORDER if any(p in CALLS[m]['species'] for p in member_pathways))]
    header = f'{title} · {n} of {n_union} genes shown' + (f' · members called by methods {", ".join(called) or "none"}' if k else '')
    listing = '; '.join(f'{i + 1} {NAMES[p]}' for i, p in enumerate(member_pathways))
    fig.text(.01, 1 - 4 / height_mm, header, ha='left', va='top', fontsize=6.5, fontweight='bold')
    fig.text(.01, 1 - 8 / height_mm, listing, ha='left', va='top', fontsize=5, wrap=True, color=MUTED)
    return fig


def program_of(pathway, labels):
    return sorted(labels.index[labels.eq(labels[pathway])]) if pathway in labels.index else [pathway]


c1_names = program_names(c1_drivers, EVIDENCE['A6·JNT'])
assert EXAMPLES['E4 · direction depends on position'] not in CALLS['A1·GSEA']['species'], 'E4 should be missed by method 1'
for rule in ('E1 · all methods agree', 'E4 · direction depends on position'):
    pathway = EXAMPLES[rule]
    members = program_of(pathway, c1_drivers)
    fig = draw_program_heatmap(members, f'{rule.split(" · ")[0]} · program "{c1_names[pathway]}" ({len(members)} pathways among the calls of method 3)')
    for ext in ('pdf', 'png'):
        fig.savefig(figures / f'fig4_gene_heatmap_{rule.split(" · ")[0]}.{ext}')
    plt.show()

In [ ]:
module3 = MODULES['B2·split']['genes'][2]
assert {'Slc22a7', 'Slc5a8', 'Cyp7b1'} <= set(module3), "method 6's module 3 should hold the S3 genes"
module3_q = MODULES['B2·split']['q'][:, 2]
enriched3 = [pathways[i] for i in np.argsort(module3_q)[:5] if module3_q[i] <= ALPHA]
top_genes = list(t_spatial[module3].nlargest(MAX_ROWS).index)
idx = gene_position[top_genes].to_numpy()
order_rows = pd.DataFrame({'mouse': species_z['mouse'][idx].argmax(axis=1), 'human': species_z['human'][idx].argmax(axis=1),
                           'gene': top_genes}).sort_values(['mouse', 'human', 'gene']).gene.tolist()
print(f"Method 6 module 3: {len(module3)} genes; {int((module3_q <= ALPHA).sum())} enriched pathways")
fig = draw_program_heatmap(enriched3, f'Method 6 module 3: {len(module3)} split-plot genes, difference falling toward S3; '
                                      f'top {MAX_ROWS} by T_spatial', rows=order_rows)
for ext in ('pdf', 'png'):
    fig.savefig(figures / f'fig4_gene_heatmap_S3_module.{ext}')
plt.show()

## 6 · Summary

**Part 1: choosing a continuous method.**
1. **The relabeling rule removes method 5**, the curve modules of the top-10% T_spatial genes. Its
   relabelings call 23 and 12 pathways against 44 for species.
2. **Calibration removes method 7**, GSEA on T_spatial. 0.139 of its relabeled p-values are ≤ 0.05
   against a nominal 0.05, and it calls 311 pathways. Its calls include all 60 of method 3's.
3. **Three methods remain, and method 3 has the most calls.** These are the joint test (method 3:
   60 calls; relabeled 2, 0; 0.060 of relabeled p ≤ 0.05), split-plot ORA (method 4: 35) and
   split-plot modules (method 6: 23). Method 3 also needs no gene cutoff and no k.
   - Methods 4 and 6 pass criterion 2 because nothing can be tested under relabeling: no gene
     passes the split-plot test there.

**Part 2: method 3 against the conventional analyses.**
4. **Whole-PT GSEA (method 1) is not specific.** It calls 158 pathways for species and 193 and 255
   under relabeling, and shares 14 calls with method 3.
5. **The S1/S2/S3 steps (method 2) agree closely with method 3, which adds pathways.** They share 47
   calls.
   - Method 3 adds 13; all are near misses for the step screen (q 0.058–0.136), and 7 of them are
     called by no specific discrete method.
   - Method 2 adds 3.
   - Method 3 also shows where inside a segment the species differ (E2) and where the direction
     changes along the PT (E4).
6. **S1/S2/S3 DE + ORA (method 8) mostly finds something else.** It passes the relabeling rule
   (118; 0, 1) but shares only 26 calls with method 3.
   - Of its 92 other calls, 66 lean towards a species offset, which method 3 is built to ignore.
     The base rate is 45% of all pathways, and none of method 3's 60 calls lean that way.
   - The other 26 lean towards shape; 9 of them have method 3 q ≤ 0.25.

Whole-PT ORA on DESeq2 lists also passes the relabeling rule (notebook 63), but it tests average
offsets only. Notebook 63 holds the full grid of 45 methods.

## 7 · Run record

In [ ]:
record = {'notebook_logic_version': NOTEBOOK_LOGIC_VERSION, 'cache_logic_version': CACHE_LOGIC_VERSION,
          'n_structures': int(len(position)), 'n_genes': len(genes), 'n_pathways': len(pathways), 'n_jobs': N_JOBS,
          'chosen': CHOSEN, 'calibration': CALIBRATION, 'methods': TABLE.reset_index().to_dict(orient='records'),
          'quadrants': QUADRANTS, 'examples': {r: NAMES[p] for r, p in EXAMPLES.items()},
          'wall_seconds': time.perf_counter() - NOTEBOOK_START}
(output / 'run_record.json').write_text(json.dumps(record, indent=2, default=str))
print(f"Notebook wall time {record['wall_seconds'] / 60:.1f} min with {N_JOBS} workers")